In [1]:
import json
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import LdaModel

from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

In [2]:
# ── CONFIG ───────────────────────────────────────────────────
DATA_PATH  = "./../data/all_bundestag_speeches_preprocessed.csv"
OUTPUT_DIR = Path("output_dictionaries")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── YOUR COLUMN NAMES ────────────────────────────────────────
TEXT_COL      = "text_preprocessed_lemmatized"  # already lemmatised ✓
FALLBACK_COL  = "text_preprocessed"             # if lemmatized is empty
DATE_COL      = "date_year"
PARTY_COL     = "Party"
SEP           = ";"                            # tab-separated


df = pd.read_csv(DATA_PATH, sep=SEP,low_memory=False)

In [3]:
# Total topics (K) chosen for the LDA model (e.g., 250 as commonly used for GermaParl)
NUM_TOPICS = 250

# Load and clean lemmatized tokens
#Assumes text_preprocessed_lemmatized is space-separated strings or list of strings
if isinstance(df["text_preprocessed_lemmatized"].iloc[0], str):
    docs = df["text_preprocessed_lemmatized"].apply(lambda x: str(x).split())
else:
    docs = df["text_preprocessed_lemmatized"]




# -------------------------------------------------------------------
# 2. BUILDING DICTIONARY & CORPUS FOR GENSIM
# -------------------------------------------------------------------
print("Building dictionary and bow corpus...")
dictionary = Dictionary(docs)
# Filter out rare and extremely frequent words to clean up topic modeling
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in docs]

Building dictionary and bow corpus...


In [ ]:


# -------------------------------------------------------------------
# 3. TRAINING LDA TOPIC MODEL
# -------------------------------------------------------------------
print(f"Training LDA Topic Model with {NUM_TOPICS} topics...")
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=10,
    iterations=25,
)

lda_model.save("./LdaModel/lda_model.gensim")

In [4]:

lda_model = LdaModel.load("./LdaModel/lda_model.gensim")

In [29]:
lda_model.show_topic(184, topn=30)

[('beschaeftigung', 0.123327985),
 ('arbeitslosigkeit', 0.11456831),
 ('tuerkisch', 0.07767049),
 ('arbeitslose', 0.06413924),
 ('arbeitskraefte', 0.06399164),
 ('arbeitslosenversicherung', 0.040277977),
 ('vermittlung', 0.035913974),
 ('arbeitslosengeld', 0.035504572),
 ('arbeits', 0.032839976),
 ('zugreifen', 0.032208793),
 ('auslaenderbehoerden', 0.026810283),
 ('arbeitskraeften', 0.024122195),
 ('inszenieren', 0.02387877),
 ('bundesanstalt', 0.023303479),
 ('kostensteigerungen', 0.02048351),
 ('arbeitsmarktes', 0.019888872),
 ('zoegerlich', 0.019728143),
 ('arbeitsplatz', 0.01833246),
 ('vermitteln', 0.0175497),
 ('eingliederung', 0.011808471),
 ('qualifizieren', 0.009797701),
 ('foerdern', 0.008646847),
 ('aufenthaltserlaubnis', 0.008145736),
 ('vertrieb', 0.008107841),
 ('arbeitsaufnahme', 0.007392146),
 ('arbeitsvermittlung', 0.0064814216),
 ('senken', 0.0063250843),
 ('bedrohen', 0.005205895),
 ('dreierlei', 0.0046040076),
 ('risikobereitschaft', 0.0045706662)]

In [ ]:

MIGRATION_TOPICS = {6,22,173,184}

def get_top_5_topic_ids(bow_doc):
        # Get topic distribution array for document
        topic_dist = lda_model.get_document_topics(
            bow_doc, minimum_probability=0.0
        )
        # Sort by probability descending and extract top 5 topic IDs
        sorted_topics = sorted(topic_dist, key=lambda x: x[1], reverse=True)[:5]
        return set(t_id for t_id, prob in sorted_topics)

    # Assign top 5 topic sets to every row
df["top_5_topics"] = [get_top_5_topic_ids(doc) for doc in corpus]

# -------------------------------------------------------------------
# 5. TAGGING SPEECHES (MIG, EU, MIG+EU)
# -------------------------------------------------------------------
print("Tagging speech categories...")
# True if ANY migration topic is present in top 5
df["is_mig"] = df["top_5_topics"].apply(
    lambda top_set: bool(top_set & MIGRATION_TOPICS)
)
df.to_csv("./results/immigration_speeches.csv")

Tagging speech categories...


OSError: Cannot save file into a non-existent directory: '../results'

## Constructing an anti-migration lexicon

In [6]:
print(f"Total Migration Speeches (mig): {df['is_mig'].sum()}")

Total Migration Speeches (mig): 24800


In [2]:
data  = pd.read_csv("./results/immigration_speeches.csv")

In [7]:
data = df[df["is_mig"]]

In [8]:
df_afd = data[data["Party"] == "AfD"].copy()

In [9]:
print(df_afd.shape)
print(df_afd["date"].min(), df_afd["date"].max())

(641, 22)
2017-11-22 2025-06-27


In [11]:
df_afd.columns

Index(['speech_identification_ent', 'date', 'period', 'session',
       'pos_speechbeginning', 'Party', 'Role', 'governing_Party', 'text',
       'text_length', 'text_preprocessed', 'text_preprocessed_lemmatized',
       'text_length_preprocessed', 'text_length_lemmatized', 'date_year',
       'date_quarter', 'similarity_expansive', 'similarity_restrictive',
       'periode_index', 'modified_flag', 'top_5_topics', 'is_mig'],
      dtype='object')

### Examining migration speech data from 2017

In [14]:

df_afd_2017 = data[
    (data["Party"] == "AfD") &
    (data["date"] < "2018-01-01")
].copy()

In [15]:
len(df_afd_2017)

6

In [27]:
df_afd_2017["text_preprocessed_lemmatized"].tolist()[0]

'dame afd verlaengerung mandat ausbildungsmission kurdistan irak urspruenglich mandat kampf islamische norden irak widmen bild hilflos mensch schnee kaelte meistens kind ausreichend kleidung barbarisch vernichtungsaktionen is berg kurdistan fluechten hilflos mensch wehrfaehig dar islamisch militaerisch besiegen kuerzlich ueberwaeltigend kurde anerkannt autonomiegebiet unabhaengig kurdistan irak ministerpraesident referendum anerkennen daraufhin militaerisch auseinandersetzung kurde armee irakisch zentralregierung massiv zunehmen drohen buergerkrieg dame pfeiler aussenpolitik waffe krisen kriegsgebiete liefern dorthin zurueckkehren berg kurdistan gefluechteten christ jesiden besiedeln flucht osten syrien ninive ebene ninive ebene kurdisch militaer kontrollieren kurdistan betrachten buergerkrieg region absehbar schlimm rueckkehr christ jesiden heimat lehnen regional katholisch orthodox bischof referendum aussenpolitisch deutschen klug agieren konfliktparteien beachten kurde tuerkei iran 

In [28]:
df_afd_2017["text"].tolist()[0]

'Sehr geehrter Herr Präsident! Sehr geehrte Damen und Herren! Die Fraktion der AfD kann der beantragten Verlängerung des Mandats für die Ausbildungsmission in Kurdistan-Irak aus folgenden Gründen nicht zustimmen:\n\nErstens. Ursprünglich war das Mandat dem Kampf gegen den „Islamischen Staat“ im Norden des Iraks gewidmet. Frau Ministerin, auch wir erinnern uns an die Bilder der hilflosen Menschen, die bei Schnee und Kälte, meistens mit Kindern und ohne ausreichende Kleidung, vor den barbarischen Vernichtungsaktionen des IS in die Berge Kurdistans geflüchtet sind. Damals war es richtig, hilflose Menschen wehrfähig zu machen. Aber heute stellt sich die Situation anders dar: Der „Islamische Staat“ ist militärisch weitgehend besiegt.\n\nGleichzeitig haben kürzlich überwältigende 92\xa0Prozent der Kurden im anerkannten Autonomiegebiet für ein unabhängiges Kurdistan gestimmt. Iraks Ministerpräsident jedoch hat das Ergebnis des Referendums nicht anerkannt. Daraufhin haben die militärischen Aus

'Sehr geehrter Herr Präsident! Sehr geehrte Damen und Herren! Die Fraktion der AfD kann der beantragten Verlängerung des Mandats für die Ausbildungsmission in Kurdistan-Irak aus folgenden Gründen nicht zustimmen:\n\nErstens. Ursprünglich war das Mandat dem Kampf gegen den „Islamischen Staat“ im Norden des Iraks gewidmet. Frau Ministerin, auch wir erinnern uns an die Bilder der hilflosen Menschen, die bei Schnee und Kälte, meistens mit Kindern und ohne ausreichende Kleidung, vor den barbarischen Vernichtungsaktionen des IS in die Berge Kurdistans geflüchtet sind. Damals war es richtig, hilflose Menschen wehrfähig zu machen. Aber heute stellt sich die Situation anders dar: Der „Islamische Staat“ ist militärisch weitgehend besiegt.\n\nGleichzeitig haben kürzlich überwältigende 92\xa0Prozent der Kurden im anerkannten Autonomiegebiet für ein unabhängiges Kurdistan gestimmt. Iraks Ministerpräsident jedoch hat das Ergebnis des Referendums nicht anerkannt. Daraufhin haben die militärischen Auseinandersetzungen zwischen den Kurden und der Armee der irakischen Zentralregierung massiv zugenommen. Da die weitere Entwicklung völlig offen ist, droht damit ein neuer Bürgerkrieg.\n\nSehr geehrte Damen und Herren, es war einmal ein begründeter Pfeiler deutscher Außenpolitik, keine Waffen in Krisen- und Kriegsgebiete zu liefern.\n\nDorthin müssen wir zurückkehren.\n\nZweitens. Die in die Berge Kurdistans geflüchteten Christen und Jesiden besiedelten vor ihrer Flucht den Osten Syriens und die Ninive-Ebene. Die Ninive-Ebene wird von kurdischen Militärs kontrolliert und als Teil eines neuen Kurdistans betrachtet. In einem möglichen Bürgerkrieg wäre diese Region absehbar mit am schlimmsten betroffen. Eine Rückkehr der Christen und Jesiden in ihre Heimat würde dann unmöglich. Daher lehnen auch die regionalen katholischen und orthodoxen Bischöfe das Referendum ab.\n\nDrittens. Außenpolitisch sollten wir als Deutsche klug agieren. Was sollten wir bei der Unterstützung der Konfliktparteien beachten? Kurden leben auch in der Türkei, im Iran und in Syrien. Das gemeinsame Ziel der Kurden ist ein eigener Staat. Wir als AfD haben grundsätzlich\xa0– das wissen Sie, meine Damen und Herren\xa0– Sympathien für Völker, die nach Souveränität und Selbstbestimmung streben.\n\nDurch unsere Ausbildung könnten auch die diplomatischen Beziehungen Deutschlands zu den Ländern Iran, Türkei und Syrien über das ohnehin bereits bestehende Maß hinaus aus diesem Grund leiden.\n\nUnsere Unterstützung der Peschmerga könnte auch nicht im Interesse der Türkei sein. Erdogan steht den Kurden jedenfalls nicht freundlich gegenüber. Er kann ein derartiges Verhalten Deutschlands, das völkerrechtlich zudem nicht durch ein UN-Mandat legitimiert ist, zutreffend als Einmischung in die inneren Angelegenheiten der Türkei verstehen. Was das Verhalten Deutschlands noch verschlimmert: Wir handeln gegen die Interessen eines Bündnispartners.\n\nAußerdem ist schon lange die Kontrolle, von wem deutsche Waffen tatsächlich verwendet wurden, unmöglich. Sie werden auf dem Schwarzmarkt gehandelt, sie werden an unterschiedliche Kämpfer weitergereicht, und sogar der „Islamische Staat“ hat zum Teil direkten Zugriff darauf.\n\nAber auch ohne Bürgerkrieg würden sich im Falle weiterer militärischer Unterstützung unsere Beziehungen zu den betroffenen Ländern in der Region und damit natürlich auch unsere diplomatischen Einflussmöglichkeiten verschlechtern. Diese Unterstützung wird ad absurdum geführt, wenn wir dadurch die sich gegenüberstehenden Parteien militärisch ertüchtigen.\n\nZudem verringern sich unsere Chancen auf eine Kooperation, zumindest auf der Arbeitsebene mit der syrischen Führung. Dabei wäre eine solche Kooperation sowohl für die Rückführung der nach Deutschland geflüchteten Syrer\n\nals auch für die verfolgten Christen in der Region sehr wichtig.\n\nBei aller notwendigen Kritik an Syrien: Für den chaldäisch-katholischen Bischof von Aleppo war und ist Syrien unter Assad der einzige Garant für die Christen, in relativer Sicherheit friedlich mit anderen Religionen zusammenleben zu können.\n\nMeine Damen und Herren, all diese Punkte sollten Sie bei Ihrer heutigen Entscheidung berücksichtigen. Deutschland hat seit Zeiten Wilhelms\xa0II. beste Beziehungen in die Region,\n\nund es wird von örtlichen Volksgruppen geachtet.\n\nKonzentrieren wir uns auf unseren diplomatischen Einfluss. Wir schlagen daher Folgendes vor: Beenden wir heute dieses Mandat! Dafür treten wir ab sofort im gesamten Konfliktgebiet des Mittleren Ostens als unparteiische und vermittelnde Macht auf.\n\nIch danke Ihnen.'


In [30]:
data["Party"].unique()

array(['Cabinet', 'FDP', 'CDU/CSU', 'SPD', 'DP', 'fraktionslos', 'GRÜNE',
       'PDS', 'LINKE', 'AfD', 'BSW'], dtype=object)

In [18]:
gruen = data[
    (data["Party"] == "GRÜNE") &
    (data["date"] < "2018-01-01")
]
len(gruen)

2041

In [25]:
gruen["text"].to_list()[5]

' Herr Präsident! Meine Damen und Herren! Liebe Freundinnen und Freunde! Grundsätzlich begrüßen wir, die GRÜNEN im Bundestag, die sozialpolitische Richtung dieses von seiten der SPD-Fraktion eingebrachten Entwurfs eines Gesetzes zur Änderung der Konkursordnung. Damit greift die SPD-Fraktion eine alte — ich betone: alte — Forderung nach Änderung der Konkursordnung auf, die nicht nur von seiten der Hamburger erhoben, sondern auch auf dem 11. Ordentlichen DGB-Bundeskongreß 1981 beschlossen wurde.\nWie aus der Begründung der SPD-Fraktion ersichtlich wird, handelt es sich hierbei lediglich um eine längst fällige Korrektur der derzeitig noch geltenden Konkursordnung, eine Korrektur, die vom Bundesarbeitsgericht bereits vorweggenommen worden ist. Wir begrüßen es grundsätzlich, wenn diejenigen, die durch den Konkurs ihres Arbeitgebers in finanzielle Schwierigkeiten geraten, ohne die noch bestehenden Rechtsunsicherheiten Ansprüche, die sich aus Sozialplänen ergeben, vorrangig geltend machen kön

' Herr Präsident! Meine Damen und Herren! Liebe Freundinnen und Freunde! Grundsätzlich begrüßen wir, die GRÜNEN im Bundestag, die sozialpolitische Richtung dieses von seiten der SPD-Fraktion eingebrachten Entwurfs eines Gesetzes zur Änderung der Konkursordnung. Damit greift die SPD-Fraktion eine alte — ich betone: alte — Forderung nach Änderung der Konkursordnung auf, die nicht nur von seiten der Hamburger erhoben, sondern auch auf dem 11. Ordentlichen DGB-Bundeskongreß 1981 beschlossen wurde.\nWie aus der Begründung der SPD-Fraktion ersichtlich wird, handelt es sich hierbei lediglich um eine längst fällige Korrektur der derzeitig noch geltenden Konkursordnung, eine Korrektur, die vom Bundesarbeitsgericht bereits vorweggenommen worden ist. Wir begrüßen es grundsätzlich, wenn diejenigen, die durch den Konkurs ihres Arbeitgebers in finanzielle Schwierigkeiten geraten, ohne die noch bestehenden Rechtsunsicherheiten Ansprüche, die sich aus Sozialplänen ergeben, vorrangig geltend machen könnten. Daß Sie, meine Herren und Damen von der CDU/CSU, das ablehnen, ist uns klar.\nWas mich allerdings verwundert, ist der Zeitpunkt, zu dem Sie, liebe Kolleginnen und Kollegen von der SPD-Fraktion, diesen Entwurf einbringen. Denn aus Ihrer eigenen Begründung geht hervor, daß die Rechtsunsicherheit bezüglich der Auslegung des Urteils des Bundesarbeitsgerichts durch die anhängigen Verfassungsbeschwerden seit nunmehr mindestens 1980 besteht, d. h. seit drei Jahren! Ich denke mir, daß es damals für Sie ein leichtes hätte gewesen sein müssen, diesen Entwurf in den Bundestag zu tragen; denn damals stellten Sie die Regierung.\nDer Zeitpunkt verblüfft also und läßt die Frage hochkommen, ob Sie es wohl wieder an der Zeit fanden, sich als Arbeitnehmerpartei profilieren zu müssen, indem Sie als Opposition längst überfällige Forderungen des DGB per Gesetzesinitiative in den\nBundestag tragen. Vermutlich wird die Gewerkschaftsführung wieder einmal, dankbar lechzend, Ihre Bemühungen aufgreifen und propagandistisch verwerten — denn zeigt Ihr wiedererwachtes Engagement für die sozialen Belange der Arbeitnehmerschaft nicht, daß Sie als verlorene Tochter wieder auf den rechten Weg sozialdemokratischer Tugend zurückgefunden haben?\nLiebe Kolleginnen und Kollegen von der SPD- Fraktion, Sie scheinen die Zeichen der Zeit nicht zu sehen bzw. geflissentlich zu übersehen, selbst wenn Sie sie hier immer wieder betonen. Von 1979 bis 1982 hat sich die Anzahl der Vergleiche und Konkurse fast verdoppelt. Das heißt in konkreten Zahlen, daß die Anzahl der Insolvenzen in drei Jahren von 8 319 auf 15 877 gestiegen ist. Glauben Sie wirklich, daß angesichts dieser drastischen Steigerung von Insolvenzen auf dem Hintergrund von andauernder Massenarbeitslosigkeit diese von Ihnen ins Rampenlicht getragene Rechtsunsicherheit — ich zitiere — „die vom Konkurs ihres Arbeitgebers betroffenen Arbeitnehmer erheblich belastet"? Oder ist es nicht vielmehr so, daß diese erhebliche Belastung weniger von der Unsicherheit herrührt, an welcher Stelle die Ansprüche aus den Sozialplänen behandelt werden, denn diese wären ohnehin nur unzureichend gedeckt, wie Sie es auch bestätigt haben, sondern daß diese erhebliche Belastung eher durch die einem Konkurs in dieser Zeit zwangsläufig folgende Arbeitslosigkeit hervorgerufen wird? Das heißt, müssen wir nicht, wenn wir eine Änderung der Konkursordnung anstreben, viel, viel weiter gehen, müssen wir nicht, wenn überhaupt, an die Wurzeln der Problematik vieler Konkurs- und Vergleichsverfahren gehen, indem wir fragen, wie zukünftig verhindert werden kann, daß Unternehmen und vor allem Konzerne Betriebsstätten aus konzernstrategischen Gründen und/oder aus Gründen der Marktaufteilung zugunsten ihrer Profitsteigerung schließen?\n— Wohlgemerkt, ich spreche hier von Betriebsschließungen und Massenentlassungen, die aus betriebswirtschaftlichen Gründen keinesfalls notwendig waren, Herr Cronenberg. Ich meine Betriebsschließungen, wie sie z. B. bei der Ulmer Farbbildröhrenfabrik Videocolor strategisch, in diesem Fall von Thomson-Brandt, vorbereitet und ohne Rücksicht auf die Betroffenen durchgeführt wurden. Hier müssen Änderungen ansetzen: bei den Mitbestimmungsrechten der Arbeitnehmer und Arbeitnehmerinnen, bei Kündigungen oder Massenentlassungen, die den Betriebsschließungen vorweggehen.\n— Eine Strategie ist kein Wert oder Unwert an sich; sie muß nicht unbedingt schlecht sein. Sollten wir also nicht in erster Linie darauf hinarbeiten, daß die Konkursordnung nicht mehr wie bislang die Schließung des Betriebs zur Folge hat, sondern auf die Erhaltung des Betriebes ausgerichtet ist?\nWir befürworten also diesen Entwurf der SPD- Fraktion, betrachten ihn allerdings angesichts der wirtschaftspolitischen Situation als einen Tropfen auf den heißen Stein. Wir streben also eine Änderung der Konkursordnung an, die insbesondere die Möglichkeit vorsehen muß, daß die in Konkurs gehenden Betriebe in das Eigentum der Belegschaft übergehen können.\nStatt Arbeitslosengeld und Sozialhilfe zu zahlen, wäre es sinnvoller, Betriebe umzustellen und als Belegschaftsbetriebe weiterzuführen.\nUnter Wettbewerbsgesichtspunkten bedeutet heutzutage jeder Konkurs, daß immer weniger Betriebe und Unternehmen den Markt unter sich aufteilen. Wer immer die freie Marktwirtschaft betont, der sollte auch dafür sorgen, daß Betriebe auch gerade dann erhalten werden, wenn sie den Belegschaften gehören und die Belegschaften selbst über Arbeit und Produktion bestimmen können,\nwobei Produktionsprogramme entwickelt werden sollten, über die nach sozialen und ökologischen Kriterien entschieden wird. — Danke sehr.\n'


### Examining migration speech data from 2018

In [16]:

df_afd_2018_migration = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2018)
].copy()


len(df_afd_2018_migration)

778

In [ ]:
df_afd_2018_migration["text_preprocessed_lemmatized"].tolist()[296]

'verehrter linke abschiebung rueckfuehrung moegen diplomatie freiwillige rueckkehr moegen syrien ungefaehr syrer genehmigung zitat veroeffentlichen beruehren essen kleidung geld fuehlen einsam ammar maarawi einzige zurueckwill angabe unhcr kehren haelfte syrische binnenfluechtlinge heimatstaedte fluechtling ausland syrien syrische aussenminister sicher rueckkehr garantieren russland syrien wiederaufnahmekapazitaeten befinden wiederaufbau andrej mahecic sprecher unhcr fluechtling rueckkehr rueckkehr wuerdevoll nachhaltig sechs plan sicher rueckkehr freiwillig rueckkehrer syrien dame abkomme syrien sicherstellen rueckkehrer unbeschadet syrien einreisen aufnehmen befriedet rueckkehrer aktivitaet flucht straftat verstoss militaerdienst verfolgen fliehen un wirksam ueberpruefen mensch freiwillig zurueckwollen mensch assad moegen herrschaft assads fakt schreien herrschaft assads fakt umgehen irgendwann rueckkehr fluechtling thematisieren fluechtling kehren unsicher abkomme kontrollieren hinw

'verehrter linke abschiebung rueckfuehrung moegen diplomatie freiwillige rueckkehr moegen syrien ungefaehr syrer genehmigung zitat veroeffentlichen beruehren essen kleidung geld fuehlen einsam ammar maarawi einzige zurueckwill angabe unhcr kehren haelfte syrische binnenfluechtlinge heimatstaedte fluechtling ausland syrien syrische aussenminister sicher rueckkehr garantieren russland syrien wiederaufnahmekapazitaeten befinden wiederaufbau andrej mahecic sprecher unhcr fluechtling rueckkehr rueckkehr wuerdevoll nachhaltig sechs plan sicher rueckkehr freiwillig rueckkehrer syrien dame abkomme syrien sicherstellen rueckkehrer unbeschadet syrien einreisen aufnehmen befriedet rueckkehrer aktivitaet flucht straftat verstoss militaerdienst verfolgen fliehen un wirksam ueberpruefen mensch freiwillig zurueckwollen mensch assad moegen herrschaft assads fakt schreien herrschaft assads fakt umgehen irgendwann rueckkehr fluechtling thematisieren fluechtling kehren unsicher abkomme kontrollieren hinweis echauffieren sparen moralkeule deal despoten erdogan saudi arabien assad rueckfuehrungsabkommen abschiebemoeglichkeit schliessen merkel dame heucheln mensch freiwillige rueckkehr syrisch fluechtling ehrlichkeit menschlichkeit fluechtling schuldig herzliche dank'

In [ ]:
df_afd_2018_migration["text"].tolist()[300]

'Sehr geehrter Herr Präsident! Werte Kolleginnen und Kollegen! Staaten, die nicht bereit sind, ihren eigenen Staatsbürgern Papiere auszustellen und so die Rückführungen ihrer eigenen Staatsbürger verhindern, muss die Entwicklungshilfe gestrichen werden.\n\nDas sagt nicht nur der gesunde Menschenverstand; das fordern wir in unserem Antrag.\n\nNach Auskunft des Bundesinnenministeriums halten sich in Deutschland über eine halbe Million Menschen mit einem abgelehnten Asylantrag auf. Nicht wenige davon können aufgrund fehlender Ausweisdokumente nicht abgeschoben werden. Sind Ausweisdokumente nicht vorhanden, dann können biometrische Daten weiterhelfen.\n\nIch habe Minister Müller bereits im Bundestag dazu befragt. Ich zitiere mit Erlaubnis des Präsidenten die Antwort des Herrn Minister:\n\nZitat Ende.\xa0– Erstaunlich fand ich übrigens, dass dieser Umstand die Kollegen der Unionsfraktionen laut Plenarprotokoll erheitert hat. Was gibt es da eigentlich zu lachen, wenn ein Entwicklungsland sch

'Sehr geehrter Herr Präsident! Werte Kolleginnen und Kollegen! Staaten, die nicht bereit sind, ihren eigenen Staatsbürgern Papiere auszustellen und so die Rückführungen ihrer eigenen Staatsbürger verhindern, muss die Entwicklungshilfe gestrichen werden.\n\nDas sagt nicht nur der gesunde Menschenverstand; das fordern wir in unserem Antrag.\n\nNach Auskunft des Bundesinnenministeriums halten sich in Deutschland über eine halbe Million Menschen mit einem abgelehnten Asylantrag auf. Nicht wenige davon können aufgrund fehlender Ausweisdokumente nicht abgeschoben werden. Sind Ausweisdokumente nicht vorhanden, dann können biometrische Daten weiterhelfen.\n\nIch habe Minister Müller bereits im Bundestag dazu befragt. Ich zitiere mit Erlaubnis des Präsidenten die Antwort des Herrn Minister:\n\nZitat Ende.\xa0– Erstaunlich fand ich übrigens, dass dieser Umstand die Kollegen der Unionsfraktionen laut Plenarprotokoll erheitert hat. Was gibt es da eigentlich zu lachen, wenn ein Entwicklungsland schafft, was die Bundesregierung nicht kann oder nicht will?\n\nLieber Herr Minister Müller\xa0– vom BMZ ist ja, glaube ich, niemand da\xa0–, wenn Sie das nächste Mal Hunderte Millionen Euro nach Marokko oder Tunesien überweisen, dann tun Sie doch den Kollegen Innenministern einen Gefallen und verlangen Sie die biometrischen Daten. Die IBAN scheint ja zu funktionieren. Zur Not schreiben Sie Ihr Anliegen in den Verwendungszweck. Da haben Sie 140\xa0Zeichen.\n\nDie Bundesregierung hat bereits 2016 in einem Akt der Verzweiflung Brandbriefe an insgesamt 17\xa0Staaten verschickt: Ägypten, Algerien, Marokko, Äthiopien, Benin, Burkina Faso, Ghana, Guinea, Guinea-Bissau, Mali, Niger, Nigeria, Tunesien, Bangladesch, Indien, Pakistan und Libanon. Und hat das Briefeschreiben was gebracht? Offensichtlich nicht.\n\nLiebe Freunde der Regierungsfraktionen, gemeinsam mit uns, der AfD, kann dieses Problem heute gelöst werden. Machen Sie heute zur Abwechslung einfach mal das Richtige!\n\nWillensbekundungen in die Richtung gab es von Ihnen schon zuhauf. Sachsens Ministerpräsident Michael Kretschmer sagte der „Frankfurter Allgemeinen Sonntagszeitung“ noch im Mai:\n\nBayerns Noch-Innenminister Joachim Herrmann sagte, dass man manchmal über die Entwicklungshilfe Druck auf Herkunftsländer machen müsse. Anfang\xa02016 ging sogar der damalige SPD-Chef Sigmar Gabriel mit der Idee hausieren, man werde nordafrikanischen Staaten die Entwicklungshilfe kürzen, wenn sie Illegale ohne Aufenthaltsrecht nicht zurücknehmen. Liebe Freunde von der SPD, da können Sie ruhig schon mal klatschen. Der Mann war immerhin Ihr Vorsitzender zu einer Zeit, als Sie noch nicht in den Umfragen hinter der AfD lagen.\n\nWas haben denn diese Herren alle gemein? Sie sind seit vielen, vielen Jahren in Regierungsverantwortung. Und was haben sie bisher gemacht? Sie haben nichts gemacht.\n\nIn einer Infratest-dimap-Umfrage vom Frühling dieses Jahres sprechen sich 59\xa0Prozent der Bürger für die Kürzung von Entwicklungshilfe bei nichtkooperativen Ländern aus. Unser Antrag sieht aber nicht einmal einen sofortigen radikalen Schnitt vor,\n\nsondern setzt auf mehrere Eskalationsstufen bis hin zur völligen Streichung der Mittel. Sie haben deshalb die einmalige Chance, Ihre Glaubwürdigkeit ein Stück weit zu reparieren. So eine Chance bekommt man nicht jeden Tag.\n\nSie wissen selbst, was passiert, wenn man zu viel \xadseehofert, also Dinge verspricht und dann nicht hält. Sie haben es am Sonntag in Bayern gesehen,\n\nund sie werden es nächste Woche Sonntag in Hessen wieder sehen. Nutzen Sie heute die Chance: Machen Sie endlich mal das Richtige!\n\nVielen Dank.'


### Examining migration speech data from 2019

In [17]:

df_afd_2019_migration = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2019)
].copy()

len(df_afd_2019_migration)

873

In [ ]:
df_afd_2019_migration["text_preprocessed_lemmatized"].tolist()[195]

'dame minute fange starten sinngemaess lucassen ausfuehrung verteidigungshaushalt rechtsstaat allgemeine erodiert druck geraten rottmann justizministerium besondere desolat zustand beschreibung bundeswehr einbeziehung justizministeriums abschliessen nahezu befinden altparteien dilcher merkel desolat zustand schlimm ddr sozialismus schlecht dorthin generaldebatte glueck einzelplan beschraenken dame etat gesamtetat kleinste vernachlaessigenswert ehemals stolze justizressort martens fuehrung durchlauferhitzer zwischenzulagernde karriereristen verkommen heiko maas schraegen justizminister schlimm dilettiert bekanntlich aussenministerium katarina barley naechst rohrkrepierer naechst rohrkrepiererin amt erzaehlen stichwort pakt rechtsstaat stichwort anwaltsverguetungen laecheln regeln weggelobt fluechten entsorgen eu schlimm lambrecht willkommen naechst sitzungswoche monat amtsantritt stunde rechtsausschuss schenken lambrecht strecken rechte hinten terminkalender schieben missachtung rechtsa

'dame minute fange starten sinngemaess lucassen ausfuehrung verteidigungshaushalt rechtsstaat allgemeine erodiert druck geraten rottmann justizministerium besondere desolat zustand beschreibung bundeswehr einbeziehung justizministeriums abschliessen nahezu befinden altparteien dilcher merkel desolat zustand schlimm ddr sozialismus schlecht dorthin generaldebatte glueck einzelplan beschraenken dame etat gesamtetat kleinste vernachlaessigenswert ehemals stolze justizressort martens fuehrung durchlauferhitzer zwischenzulagernde karriereristen verkommen heiko maas schraegen justizminister schlimm dilettiert bekanntlich aussenministerium katarina barley naechst rohrkrepierer naechst rohrkrepiererin amt erzaehlen stichwort pakt rechtsstaat stichwort anwaltsverguetungen laecheln regeln weggelobt fluechten entsorgen eu schlimm lambrecht willkommen naechst sitzungswoche monat amtsantritt stunde rechtsausschuss schenken lambrecht strecken rechte hinten terminkalender schieben missachtung rechtsausschusses ausdruecken plan amt lambrecht monat amt verdaddelt schwerpunkt lagen absolution schulschwaenzer zustaendig hektisch unausgegoren ankuendigung waffenrecht zustaendig ansonsten museum rechtsstaat aeussern ressort lambrecht visitenkarte rechtsstaats gewaltmonopol unabhaengig justiz rechtsstaat lambrecht bundesjustiz verfassungsministerin millionenfacher rechtsbruch fehlend rechtsdurchsetzung ueberwiegen illegal migration aeussern lambrecht richter gericht flut importiert kriminalitaet million straftat zuwanderer ueberrollen aeussern lambrecht lambrecht hunderttausende rechtsstaatswidrig vollzogen abschiebung aeussern lambrecht zigtausende vollzogen haftbefehle aeussern lambrecht staerkung rechtsstaats pakt rechtsstaat aeussern lambrecht erosion rechtsstaats zunehmend durchbrechung gewaltenteilung auswahl ober richter bundesverfassungsrichter parteibuch parteienkluengel bruch europarechts aeussern lambrecht stunde lambrecht charmante gesicht exekutive rechtsstaat stolz anwalt rechtsstaats stoppen erosion rechtsstaats dank'

In [ ]:
df_afd_2019_migration["text"].tolist()[200]

'Sehr geehrte Frau Präsidentin! Werte Kollegen! Vergeblich haben wir, die AfD, immer wieder Statistiken und Auswertungen zu flüchtlingsbedingten Kosten im Zusammenhang mit dem Gesundheitsfonds gefordert.\n\nGeschehen ist nichts und kann es auch derzeit nicht. Denn es gibt keine aussagefähigen Zahlen, die als Grundlage für den Gesundheitsfonds dienen können.\n\nWoran liegt das? Diese Große Koalition hat jahrelang zugelassen, dass Krankenkassen und Krankenhäuser unzulässige pauschale Rechnungskürzungen\xa0– wahrscheinlich insgesamt in Höhe mehrerer Milliarden\xa0– vereinbaren und auf diese Weise Abrechnungsprüfungen umgehen. Das ist ein bodenloser Skandal auf Kosten der Beitragszahler, der Krankenkassen und des Steuerzahlers.\n\nDer Bundesrechnungshof und das Bundesversicherungsamt und die Aufsichtsbehörden der Länder haben dann auch endlich im November 2018 die Rechtswidrigkeit dieser Vereinbarungen ausdrücklich bestätigt. Über Jahre kamen Krankenkassen ihrer Pflicht zur Prüfung der Kra

'Sehr geehrte Frau Präsidentin! Werte Kollegen! Vergeblich haben wir, die AfD, immer wieder Statistiken und Auswertungen zu flüchtlingsbedingten Kosten im Zusammenhang mit dem Gesundheitsfonds gefordert.\n\nGeschehen ist nichts und kann es auch derzeit nicht. Denn es gibt keine aussagefähigen Zahlen, die als Grundlage für den Gesundheitsfonds dienen können.\n\nWoran liegt das? Diese Große Koalition hat jahrelang zugelassen, dass Krankenkassen und Krankenhäuser unzulässige pauschale Rechnungskürzungen\xa0– wahrscheinlich insgesamt in Höhe mehrerer Milliarden\xa0– vereinbaren und auf diese Weise Abrechnungsprüfungen umgehen. Das ist ein bodenloser Skandal auf Kosten der Beitragszahler, der Krankenkassen und des Steuerzahlers.\n\nDer Bundesrechnungshof und das Bundesversicherungsamt und die Aufsichtsbehörden der Länder haben dann auch endlich im November 2018 die Rechtswidrigkeit dieser Vereinbarungen ausdrücklich bestätigt. Über Jahre kamen Krankenkassen ihrer Pflicht zur Prüfung der Krankenhausabrechnungen nicht nach. Sie hatten individuelle Vereinbarungen mit den Krankenhäusern über pauschale Rechnungskürzungen beschlossen und im Gegenzug auf Abrechnungsprüfungen verzichtet. Damit unterblieben auch die für bestimmte Fälle gesetzlich vorgeschriebenen Prüfungen durch den Medizinischen Dienst der Krankenversicherung. Krankenkassen sind aber gesetzlich verpflichtet, eine gutachtliche Stellungnahme des Medizinischen Dienstes der Krankenversicherung einzuholen, wenn dies nach Art, Schwere, Dauer oder Häufigkeit der Erkrankung oder nach dem Krankheitsverlauf erforderlich ist.\n\nMit anderen Worten: Diese Vereinbarungen ermöglichen es Krankenhäusern, sich von Prüfungen durch die Krankenkassen und damit des Medizinischen Dienst freizukaufen. Wer aber hindert die Krankenhäuser, die Abzüge im Vorfeld einzukalkulieren und überhöhte Rechnungen auszustellen, vor allem, wenn sie wissen, dass eine Überprüfung ohnehin nicht stattfindet? Es wurde also ein System erschaffen und von dieser Bundesregierung jahrelang geduldet, das millionenfache Gelegenheit zum Abrechnungsbetrug ermöglicht.\n\nEs ist ein System, in dem durch pauschalen Verzicht der Überprüfung der Krankenhausabrechnungen in Kauf genommen wurde, dass medizinische Fälle unerkannt bleiben, in denen der Medizinische Dienst eingeschaltet werden müsste. Mit anderen Worten: Derzeit haben weder die Krankenkassen noch die Bundesregierung einen genauen Überblick, mit welchen Krankheiten unsere Bevölkerung in welchem Umfang wirklich zu tun hat\xa0\nein Skandal und eine kaum zu übertreffende Verantwortungslosigkeit jedem einzelnen Bürger unseres Landes gegenüber.\n\nUnd es geht weiter: Da die so gewonnenen Daten die Grundlage für die Zuweisung aus dem Gesundheitsfonds bilden, sind alle Berechnungen, die dem Gesundheitsfonds derzeit zugrunde liegen, unbrauchbar, Makulatur, schlicht und einfach für die Katz.\n\nUnd was macht diese Bundesregierung? Sie lässt sich Zeit. Erst am 17.\xa0Juli 2019, also in der Sommerpause, wurde vom Kabinett der Entwurf eines Reformgesetzes beschlossen, um eine gesetzliche Klarstellung des Verbots dieser im November 2018 endlich als unzulässig erkannten Vereinbarung in die Wege zu leiten. Jahrealte Abrechnungen müssen jetzt überprüft werden, um Abrechnungsfehler und vor allem Betrügereien zu entdecken, die zu einem Schaden von vielen Milliarden Euro geführt haben können.\n\nWie viele Millionen kostet diese Aufarbeitung den Steuerzahler\xa0– denn wer sonst soll das alles bezahlen? Wie können Sie von Bürgern Gesetzestreue erwarten, wenn Sie selbst so grob fahrlässig mit der Gesundheit und dem Geld unserer Bürger umgehen? Wie können Sie bei dieser Sach- und Rechtslage zuverlässige Aussagen über den Finanzbedarf des Gesundheitsfonds treffen?\n\nLast, but not least: Nach den neuesten Zahlen des Robert-Koch-Instituts sind Erkrankungen an Hepatitis B im letzten Jahr weiter explosiv gestiegen. Hepatitis B zählt bei chronischem Verlauf zu den bedeutendsten Ursachen von Leberzellkarzinomen,\n\nund der Tod als Folge hiervon rangiert weltweit auf Platz zwei der krebsbedingten Todesursachen.\n\nRechnet man die diesjährigen Zahlen des RKI hoch, kommt man auf 5\xa0560 Fälle. Das bedeutet von 2018 zu 2019 eine Steigerung von circa 20\xa0Prozent, also eine Steigerung von circa 635\xa0Prozent seit 2014, als es in Deutschland nur 755\xa0Fälle gab. Bei Asylsuchenden kamen 2017\xa0\xa062\xa0Prozent aus Afrika und 29\xa0Prozent aus Asien, vorwiegend aus Syrien und Afghanistan.\n\nAktuelle Studien aus Deutschland zeigen laut RKI für Personen mit Migrationshintergrund, dass 80\xa0Prozent ihrer Erkrankungen an einer aktiven Hepatitis\xa0B unbekannt war und dass sie auch nicht wussten, wie Hepatitis\xa0B übertragen wird.\n\nWir, die AfD, fordern daher gezielte Screeningmaßnahmen bei Asylsuchenden;\n\ndenn die Dunkelziffer der Infizierten birgt wie bei HIV und Tuberkulose ein schreckliches epidemiologisches Potenzial.\n\nOder sollen solche Fälle wie der im September 2017 in Dresden gehäuft auftreten, Herr Spahn, wo nach der Entdeckung eines aktiven TB-Falls circa 2\xa0000 Kontaktpersonen ermittelt wurden und über 3\xa0000 Blutentnahmen erfolgt sind? Damals wurden 120 latente tuberkulöse Infektionen und sieben aktive Tuberkulosen ermittelt.\n\nWir, die AfD, brechen diese Schweigespirale. Unsere Bevölkerung muss vor derartigen Krankheiten geschützt werden, und das nicht nur wegen der exorbitant hohen Kosten.\n\nUnd wie oft müssen wir, die AfD, noch auf diese für jedermann offenkundigen Gefahren hinweisen, bis auch Sie, die Sie hier alle sitzen, endlich handeln?'












































































### Examining migration speech data from 2020

In [18]:

df_afd_2020_migration = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2020)
].copy()


len(df_afd_2020_migration)

889

In [ ]:
df_afd_2020_migration["text_preprocessed_lemmatized"].tolist()[286]

'dame lieben zuschauer bildschirmen einzelplan umfangreiche haushaltsberatungen umfassten inner staerken asyl migration massiv absenken sport schwierig neueinstellungen sicherheitsbehoerden eklatanten versaeumnis vorgaengerregierungen nachholen afd modernisierung sicherheitsstruktur modern ausgebildet grenze innere material personal aufstocken seehofer aufstocken wille ursache gewachsen bedrohungslage effizienter grenzschutz asylmissbrauch importieren kriminalitaet fehlanzeige islamistische gefaehrder grotesk aufwand ueberwachen anstatt komplett abschiebehaft praeventivgewahrsam unfassbar teuer staatlich alimentierung linksextremisten aufsetzen bekaempfung gewaltbereiten islamismus verstaerken bekaempfung jeglich extremismus genehmen demonstration polizei diffamieren bejubeln einsatz wasserwerfern reizgas demo einschraenkung grundrechte schaden ansehen ordnungskraefte fakt polizei spielball rueckhalt verlieren rueckendeckung seehofer heimat begriff bieten heimatschutz massnahme aktiv v

'dame lieben zuschauer bildschirmen einzelplan umfangreiche haushaltsberatungen umfassten inner staerken asyl migration massiv absenken sport schwierig neueinstellungen sicherheitsbehoerden eklatanten versaeumnis vorgaengerregierungen nachholen afd modernisierung sicherheitsstruktur modern ausgebildet grenze innere material personal aufstocken seehofer aufstocken wille ursache gewachsen bedrohungslage effizienter grenzschutz asylmissbrauch importieren kriminalitaet fehlanzeige islamistische gefaehrder grotesk aufwand ueberwachen anstatt komplett abschiebehaft praeventivgewahrsam unfassbar teuer staatlich alimentierung linksextremisten aufsetzen bekaempfung gewaltbereiten islamismus verstaerken bekaempfung jeglich extremismus genehmen demonstration polizei diffamieren bejubeln einsatz wasserwerfern reizgas demo einschraenkung grundrechte schaden ansehen ordnungskraefte fakt polizei spielball rueckhalt verlieren rueckendeckung seehofer heimat begriff bieten heimatschutz massnahme aktiv vorangetrieben haushalt miete ballungsraeumen polizeiabsolventen zwingen wohngemeinschaft bilden anfahrtswege dienststellen kauf jung absolventen grossstadt ortszuschlag einfuehren wohnraum bundesbedienstete bestand zustaendig wohnfuersorge bundesfinanzministerium abschieben wohnungsbau steigerung anstrengung abschluss ueppig ausstattung stiftung verwundern bereinigungssitzung globalzuschuesse stiftung million erhoehen nullen haushaltsausschuss mittelbedarf zustande nullen ueppig mittelausstattung stiftung intransparenter prozess mitwirken profitieren lehnen haushalt'



In [ ]:
df_afd_2020_migration["text"].tolist()[303]

'Ich bemühe mich, Frau Präsidentin.\xa0– Frau Präsidentin! Werte Kolleginnen und Kollegen! Wir sprechen heute über einen verbesserten Zugang zu Teilhabeleistungen. Das sind Leistungen, die Menschen mit Behinderungen dabei helfen, ihr Leben zu meistern.\n\nSo wichtig es ist, über bessere Regelungen in der Teilhabe zu reden, so wichtig ist es, dabei fair und gerecht zu bleiben. Menschen mit Behinderungen haben es verdient, dass sie nicht zum Spielball von Ideologie werden, sondern dass die Politik ihre Bedürfnisse ernst nimmt. Und darum geht es Ihnen von den Grünen leider nicht.\n\nFrau Rüffer, was Sie vorgetragen haben, ist unglaubwürdig, wenn man das mit dem Antrag vergleicht, den Sie gestellt haben.\n\nDer Antrag bringt keine echten Lösungen; er fabuliert an verschiedenen Stellen, dass etwas verbessert werden soll, sagt aber nicht, wie. Vieles von dem, was Sie vorschlagen, existiert schon. Was ich Ihnen wirklich vorwerfe, ist die Tatsache, dass es keinen einzigen Antrag der Grünen zu 

'Ich bemühe mich, Frau Präsidentin.\xa0– Frau Präsidentin! Werte Kolleginnen und Kollegen! Wir sprechen heute über einen verbesserten Zugang zu Teilhabeleistungen. Das sind Leistungen, die Menschen mit Behinderungen dabei helfen, ihr Leben zu meistern.\n\nSo wichtig es ist, über bessere Regelungen in der Teilhabe zu reden, so wichtig ist es, dabei fair und gerecht zu bleiben. Menschen mit Behinderungen haben es verdient, dass sie nicht zum Spielball von Ideologie werden, sondern dass die Politik ihre Bedürfnisse ernst nimmt. Und darum geht es Ihnen von den Grünen leider nicht.\n\nFrau Rüffer, was Sie vorgetragen haben, ist unglaubwürdig, wenn man das mit dem Antrag vergleicht, den Sie gestellt haben.\n\nDer Antrag bringt keine echten Lösungen; er fabuliert an verschiedenen Stellen, dass etwas verbessert werden soll, sagt aber nicht, wie. Vieles von dem, was Sie vorschlagen, existiert schon. Was ich Ihnen wirklich vorwerfe, ist die Tatsache, dass es keinen einzigen Antrag der Grünen zu geben scheint, in dem nicht irgendwo Politik für Migranten versteckt ist.\n\nIn diesem spezifischen Antrag fabulieren Sie sogar Menschenrechte herbei, die es nicht gibt, um Politik gegen Deutschland zu begründen. Das ist in der Behindertenpolitik besonders verwerflich, meine Damen und Herren!\n\nUnd darauf will ich eingehen: Auf Seite\xa08 ersinnen Sie, das Recht auf Teilhabe sei ein universelles Menschenrecht, um dann gleich hinzuzufügen, dass es nicht vom aufenthaltsrechtlichen Status abhängig gemacht werden darf.\n\nDamit begründen Sie ein unbegrenztes Leistungs-, Wunsch- und Wahlrecht. Die Gemeinschaft hat alles zu bezahlen für alle aus aller Welt. Die Wahrheit ist aber\xa0– um das jetzt ein für alle Mal zu klären\xa0–: Ein universelles Menschenrecht auf Teilhabe gibt es nicht, übrigens genauso wenig wie auf Sozialleistungen oder Migration,\n\nnicht in der Europäischen Menschenrechtskonvention und auch nicht in der Allgemeinen Erklärung der Menschenrechte der UN. Dort gibt es aber eine gegenseitige Verantwortlichkeit zwischen Gemeinschaft und Individuum. Lesen Sie mal Artikel\xa022: Als Mitglied der Gesellschaft hat jeder unter Berücksichtigung der Mittel des Staates Anspruch darauf, in den Genuss der Rechte zu gelangen, die für seine Würde unentbehrlich sind.\xa0– Auf der anderen Seite gibt es Artikel\xa029: „Jeder hat Pflichten gegenüber der Gemeinschaft.“\xa0– Ja, schreiben Sie sich das mal hinter die grünen Ohren!\n\nUnd in der UN-Behindertenrechtskonvention gibt es auch keine entsprechenden Regelungen. In Artikel\xa019 haben die Vertragsstaaten deklariert, dass sie Maßnahmen treffen, um Menschen mit Behinderungen ihre volle Einbeziehung in die Gemeinschaft und Teilhabe an der Gemeinschaft zu erleichtern. Das steht aber unter dem Progressionsvorbehalt des Artikels\xa04, nämlich der verfügbaren Mittel und der Verwirklichung nach und nach.\n\nDer Anspruch auf Leistungen ist also kein universelles Menschenrecht. Der Staat hat die Pflicht, im Rahmen seiner Mittel für jedes Mitglied der Gesellschaft auf Teilhabe hinzuwirken. Ihre Prämisse, die Sie in den Antrag geschrieben haben, ist einfach falsch, wie in vielen anderen Anträge auch. Und das werfe ich Ihnen vor: dass Sie die Behindertenpolitik zu dieser Art von Forderungen missbrauchen.\n\nDie anderen wohlfeilen Forderungen, die Sie stellen, fallen ganz schnell auseinander, wenn man sich intensiver damit beschäftigt. Ich mache mal einen kurzen Durchlauf.\n\nErstens. Das Wunsch- und Wahlrecht wollen Sie ausweiten, insbesondere bezüglich der Leistungen und des Wohnorts und des Pflegetyps. Nach der geltenden Rechtslage gibt es bereits jetzt einen Ausschluss deutlich teurerer Leistungen nur nach einer Abwägung der Situation, nach einer Prüfung der Zumutbarkeit. Das steht in §\xa0104 SGB\xa0IX; den muss man halt lesen.\n\nZweitens. Sie wollen neue Berichtspflichten und neue Fristen einführen. Sie haben selber festgestellt, dass die Behörden schon überlastet sind; Sie wollen sie aber noch weiter überfordern, obwohl es solche Fristen schon längst gibt. Bei Überschreitungen gibt es die sogenannte Genehmigungsfiktion; das heißt, die Leistungsberechtigten könnten sich die Leistungen einfach selber einkaufen. Das stört Sie aber wieder. Sie sagen, das könnten die nicht finanzieren, haben aber offensichtlich überlesen, dass diese Leute Anspruch auf Abschlagszahlungen haben, und zwar nach §\xa018 Absatz\xa04 SGB\xa0IX.\n\nGanz wichtig sind Ihnen natürlich die Leistungen für Asylbewerber. Sie fordern Eingliederungshilfen vom ersten Tag an. Wollen wir wirklich den eingliedern, der geduldet ist oder direkt vor der Abschiebung steht?\xa0– Das ist ganz bestimmt nicht das Ziel von Behindertenrechtspolitik.\n\nMeine Damen und Herren, der Antrag, den Sie gestellt haben, dient nicht den Menschen mit Behinderungen. Er dient dazu, die Tür zu öffnen für Eingliederung und Extraleistungen für Menschen aus der ganzen Welt. Teilhabe wird so erweitert zu einer Art Nachteilsausgleich für illegale Migranten. Wenn Sie so die Menschen mit Behinderungen für Ihre One-World-Ideologie\xa0–'























### Examining migration speech data from 2022

In [19]:

df_afd_2022_migration = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year == 2022)
].copy()

len(df_afd_2022_migration)

909

In [ ]:
df_afd_2022_migration["text_preprocessed_lemmatized"].tolist()[248]

'dame krankenhauspflege erschreckend jahrelang ignoranz herrschend fundamental pflegemassnahmen geboten aufwand durchfuehren personal ausbildung mangeln impliziten rationierung pflege patient ueberwachung patient gespraech angehoerige korrekt dokumentation anmessen pflegearbeit zunehmen aushoehlen qualifizieren ausgebildet pflegekraefte uebernehmen personal gewaehrleisten krankenschwester kranke kuemmern hol bringedienst reinigungskraft hauptberufliche dokumentationskraft resultierend unzufriedenheit muenden flucht qualifiziert beruf ungerecht corona pflegebonusgesetz mitarbeiter leeren unmut pflege krankenhausbereich teufelskreis durchbrechen frage personal akquirieren vortrag ueblich zurufe links block zuwanderung heissen krankenhauspflege rot gruen zuwanderung psychologisch werte zuwanderung jahrzehnt kilimandscharo auftuermen attraktiv zuwanderungsland auslaendische fachkraefte steuer abgabe arbeitsbedingungen schlecht attraktivste zuwanderungsland sozialfluechtlinge europaeisch na

'dame krankenhauspflege erschreckend jahrelang ignoranz herrschend fundamental pflegemassnahmen geboten aufwand durchfuehren personal ausbildung mangeln impliziten rationierung pflege patient ueberwachung patient gespraech angehoerige korrekt dokumentation anmessen pflegearbeit zunehmen aushoehlen qualifizieren ausgebildet pflegekraefte uebernehmen personal gewaehrleisten krankenschwester kranke kuemmern hol bringedienst reinigungskraft hauptberufliche dokumentationskraft resultierend unzufriedenheit muenden flucht qualifiziert beruf ungerecht corona pflegebonusgesetz mitarbeiter leeren unmut pflege krankenhausbereich teufelskreis durchbrechen frage personal akquirieren vortrag ueblich zurufe links block zuwanderung heissen krankenhauspflege rot gruen zuwanderung psychologisch werte zuwanderung jahrzehnt kilimandscharo auftuermen attraktiv zuwanderungsland auslaendische fachkraefte steuer abgabe arbeitsbedingungen schlecht attraktivste zuwanderungsland sozialfluechtlinge europaeisch nachbar freund fuellen sonderzug mensch potenzial million zuwanderer genuegen fachkraefte sortierung koffer flughafen gewinnen geschweige ausgebildet pflegekraefte absolut unverzichtbar sprachkenntnissen dazugehoeren moegen explizit bundesweite studie pflegen arbeitnehmerkammer bremen befragte erwaehnen besagen vollzeitpflegekraefte rueckkehr beruf aufstockung arbeitszeit sofern arbeitsbedingungen pflege personal mangeln tausende einrichtungsbezogene impfpflicht beruf abschrecken taetigen einstellungsstau rechtswidrige einrichtungsbezogene impfpflicht laufen aussetzen gesundheits pflegewesen kernaufgaben begreifen unausweichlich pflegekatastrophe schlittern verantwortlich vorbildlich gesundheitswesen beschaedigen geld gesundheitssystem dank'


In [ ]:
df_afd_2022_migration["text"].tolist()[261]

'Sehr geehrte Frau Präsidentin! Sehr geehrte Damen und Herren! Herr Grötsch, die Bundespolizei nicht politisch zu instrumentalisieren – wir nehmen Sie und die Regierung beim Wort; immer wieder gern.\n\nDie Beschäftigung mit der inneren Sicherheit darf für die Politik niemals zum Selbstzweck werden. Vor fast sechs Jahren, am 19. Dezember 2016, tötete der polizei- und nachrichtendienstlich bekannte islamistische Attentäter Anis Amri auf dem Berliner Breitscheidplatz insgesamt 13 Menschen und verletzte 67 Personen. Solche Gräueltaten dürfen sich nie wieder wiederholen. In drei Tagen werden wir wieder in angemessener Form der Opfer dieser schrecklichen Tat gedenken. Ich bin in Gedanken bei den Verwandten der Opfer.\n\nEin unverzichtbarer Bestandteil des Schutzes der öffentlichen Sicherheit, aber tatsächlich auch des Schutzes der nationalen Grenzen, der Schienenwege und der Flughäfen ist unsere Bundespolizei. Circa 54\u202f000 Mitarbeiter sind in diesem Bereich tätig. Von 2017 bis 2021 hat 

'Sehr geehrte Frau Präsidentin! Sehr geehrte Damen und Herren! Herr Grötsch, die Bundespolizei nicht politisch zu instrumentalisieren – wir nehmen Sie und die Regierung beim Wort; immer wieder gern.\n\nDie Beschäftigung mit der inneren Sicherheit darf für die Politik niemals zum Selbstzweck werden. Vor fast sechs Jahren, am 19. Dezember 2016, tötete der polizei- und nachrichtendienstlich bekannte islamistische Attentäter Anis Amri auf dem Berliner Breitscheidplatz insgesamt 13 Menschen und verletzte 67 Personen. Solche Gräueltaten dürfen sich nie wieder wiederholen. In drei Tagen werden wir wieder in angemessener Form der Opfer dieser schrecklichen Tat gedenken. Ich bin in Gedanken bei den Verwandten der Opfer.\n\nEin unverzichtbarer Bestandteil des Schutzes der öffentlichen Sicherheit, aber tatsächlich auch des Schutzes der nationalen Grenzen, der Schienenwege und der Flughäfen ist unsere Bundespolizei. Circa 54\u202f000 Mitarbeiter sind in diesem Bereich tätig. Von 2017 bis 2021 hat sich die Anzahl der Planstellen bei der Bundespolizei von 42\u202f000 auf circa 50\u202f000 erhöht, der jährliche Haushalt ist im selben Zeitraum von 3,3 Milliarden auf 4,7 Milliarden Euro angehoben worden. Jeder einzelne Cent, den wir in unsere Bundespolizei investiert haben, ist gut angelegtes Geld für die Zukunft Deutschlands. Die Bundespolizei muss auch zukünftig weiter gestärkt und ausgebaut werden.\n\nAber es bleibt nach wie vor viel zu tun. Als Abgeordneter habe ich in den sitzungsfreien Wochen mehrfach Polizeidienststellen aufgesucht und mich mit Dienststellenleitern und Beamten des Polizeivollzugsdienstes unterhalten.\n\n– So fleißig sind wir. – Hierbei konnte ich mir ein eigenes Bild von den Gegebenheiten und den Anforderungen in den täglichen Abläufen bei der Bundespolizei machen. Einzelne Schwerpunkte sind dabei immer wieder zur Sprache gekommen:\n\nZur Bekämpfung der Schleuserkriminalität und zur Ahndung der Einreisestraftaten sollte es den Beamten zukünftig möglich sein, auch außerhalb der 30-Kilometer-Zone im gesamten Bundesgebiet eigenständig ermitteln zu dürfen.\n\nAuch die Endsachbearbeitung der von der Bundespolizei zuerst festgestellten Straftaten ist ein wichtiger Beitrag, um behördliche Ressourcen zu bündeln. Die derzeit noch gängige Praxis, eine Strafakte quasi für die Staatsanwaltschaft komplett vorzubereiten, um sie dann der Landespolizei zu übergeben, widerspricht modernen Effektivitätsgrundsätzen. Hier müssen dringend rechtliche Rahmenbedingungen geändert werden.\n\nDie Privatisierung der Autobahnen hat insbesondere in Grenzregionen dazu geführt, dass die Einrichtung von größeren Kontrollstellen nicht mehr unangekündigt möglich ist, sondern zuvor bei der privaten Betreibergesellschaft angemeldet werden muss. Die darauf folgenden Stauanzeigen auf den Navigationsgeräten der Fahrzeuge leiten den zu kontrollierenden Verkehr dann regelmäßig an der Kontrollstelle vorbei. Die Nutzung des Überraschungseffekts wird hierdurch deutlich eingeschränkt.\n\nEs bräuchte auch vermehrt sogenannte Verkehrstrichter, um die Geschwindigkeit des Fahrverkehrs bei derartigen Kontrollen des Verkehrs auf der Bundesautobahn zu verlangsamen. Eine solche Mittelbeschaffung wäre ein wichtiger Beitrag zum Arbeitsschutz unserer eingesetzten Beamten.\n\nBei für einen längeren Zeitraum eingerichteten Kontrollstellen, beispielsweise auf Autobahnen, benötigen die Beamten insbesondere an kalten Wintertagen, aber auch zum Schutz vor Regen und Nässe mobile Kontrollstellen mit entsprechender Ausstattung. Praxisbezogene Bedarfe sind beispielsweise sogenannte Agrarzelte, welche als Überdachung bei Kontrollen von Reisebussen erforderlich sind, damit die Fahrgäste und Beamten bei Wind und Wetter nicht im Freien stehen. Summa summarum benötigt die Bundespolizei zukünftig vermehrt lageangepasste Einsatzmittel, um Arbeitsweisen optimieren zu können.\n\nEines der größten Probleme im Dienstalltag der Bundespolizei bleibt aber die viel zu geringe Zahl an Rückführungen von Personen, die sich unerlaubt in Deutschland aufhalten. Im Jahr 2021 waren 33\u202f600 Rückführungen von Ausländern vorgesehen. Allerdings wurde davon nur die Hälfte, also rund 15\u202f000 Rückführungen, auch wirklich vollzogen. Der Jahresbericht der Bundespolizei sagt aus: Für die Diskrepanz an Rückführungen war hauptursächlich, dass die zur Abschiebung vorgesehenen Personen der Bundespolizei am Flugtag nicht zur Rückführung übergeben werden konnten. – Hier müssten Sie, Frau Innenministerin, ansetzen und endlich eine wirksame Abschiebeoffensive einleiten,\n\nanstatt Ihre Energie für die Jagd auf Phantombedrohungen zu verschwenden.\n\nWährend die Anzahl der im Jahr 2021 im Zuständigkeitsbereich der Bundespolizei festgestellten Straftaten in vielen Bereichen rückläufig war, lagen die größten Zunahmen im Bereich Betrug und bei Verstößen gegen das Aufenthaltsgesetz. Sage und schreibe 171\u202f000 Verstöße gegen das Aufenthaltsgesetz hat allein die Bundespolizei im Jahr 2021 festgestellt. Dies ist nicht zuletzt auf die Politik der offenen Grenzen unserer Bundesregierung zurückzuführen.\n\nLeider schweigt sich der Antrag der CDU dazu aus, wie Sie die genaue Zuständigkeitsverteilung bei der Strafverfolgung durch Land und Bund voneinander abgrenzen wollen. Eine Doppelzuständigkeit von Bundes- und Landespolizei zur Durchführung eines strafprozessualen Ermittlungsverfahrens etwa bei Taten, die an Bahnhöfen begangen werden, würde zu einem unüberschaubaren Behördenchaos führen. Das Thema Strafverfolgung ist aber zu bedeutend, als dass man es hier schaufensterartig in nur drei Zeilen abschließend darstellen könnte.\n\nZum Schluss sei noch gesagt: Beamte, insbesondere Polizeibeamte, sind Recht und Gesetz verpflichtet und dürfen nicht den politischen Fantasien der jeweiligen Regierung unterliegen. Nicht nur die Bundespolizisten gehören vor dem Generalverdacht geschützt, sondern auch neue Bewerber für den Polizeivollzugsdienst. Einen weiteren politischen Gesinnungs-TÜV von Nancy Faesers Gnaden lehnen wir entschieden ab.\n\nIch wünsche allen Deutschen einen friedlichen vierten Advent und frohe Weihnachten.'








































































### Examining migration speech data from 2024,2025

In [21]:

df_afd_2024_migration = data[
    (data["Party"] == "AfD") &
    (data["date"].dt.year >2023)
].copy()

len(df_afd_2024_migration)

1195

In [ ]:
df_afd_2024_migration["text_preprocessed_lemmatized"].tolist()[485]

'dame vordergruendig einbuergerung sonderfall option geringfuegig modifizierung vorausgegangen ausweitung zugriff staatsbuergerschaft ampel kuerzen frist einbuergerungsanspruch acht sonderfall zuvor fristenverkuerzung regelhafter akzeptanz doppelt staatsbuergerschaft heissen staatsbuergerschaft konsumartikel preis nullen absitzen warten turboeinbuergerung sonderfall beibehalten unfassbarer dreistigkeit taeuschen dame einbuergerung integriert integriert heissen hiesig leitkultur verinnerlichen heimat loyal stattdessen zerfallen zusehends regeln einbuergerung bleibt akzeptanz regelhaften doppeln staatsbuergerschaft bleibt generell einbuergerung frueh einbuergerung ermessen bleibt staatsbuergerschaft schleichen geburt frueh abstammung bleibt bleibt bleibt bleibt bleibt desaster totalversagen dame vollstaendig beibehaltung kosmetische korrektur absurd spezialregelung koalitionsvertrag durchsetzen unterwuerfig akzeptieren verkaufen grundsaetzliche wende einbuergerung behauptung unwahr wisse

'dame vordergruendig einbuergerung sonderfall option geringfuegig modifizierung vorausgegangen ausweitung zugriff staatsbuergerschaft ampel kuerzen frist einbuergerungsanspruch acht sonderfall zuvor fristenverkuerzung regelhafter akzeptanz doppelt staatsbuergerschaft heissen staatsbuergerschaft konsumartikel preis nullen absitzen warten turboeinbuergerung sonderfall beibehalten unfassbarer dreistigkeit taeuschen dame einbuergerung integriert integriert heissen hiesig leitkultur verinnerlichen heimat loyal stattdessen zerfallen zusehends regeln einbuergerung bleibt akzeptanz regelhaften doppeln staatsbuergerschaft bleibt generell einbuergerung frueh einbuergerung ermessen bleibt staatsbuergerschaft schleichen geburt frueh abstammung bleibt bleibt bleibt bleibt bleibt desaster totalversagen dame vollstaendig beibehaltung kosmetische korrektur absurd spezialregelung koalitionsvertrag durchsetzen unterwuerfig akzeptieren verkaufen grundsaetzliche wende einbuergerung behauptung unwahr wissentlich unwahr verdeckungspropaganda verhandlungsversagens gescheitert migrationswende verkaufen waehlertaeuschung dame verleihung staatsbuergerschaft ausweis angekommenseins heimat verstreichen integriertheit einbuergerung knapp million illegal eingedrungen migranten schwarz rot einbuergerungsanspruch schenken illegal migranten nahost afrika bestand verfolgung mehrheitlich nachbarland zig sicher vorteilsnahme mehrheitlich grossangelegten asylbetrug dame personenkreis angestammt kultur prosperitaet ueppig sozialsysteme kalten zyniker macht organisation per raschester einbuergerung antideutsche mangelnd qualifikation staatsabhaengiges prekariat importieren notgedrungen umverteilungsparteien waehlen einbuergerungspolitik aufforstungsprogramm rot rot gruene waehlerschaft wusste umfrage muslime auslaender angriff staatsvolk par excellence dame ansaessige langem kinderquote reproduktionsfaktor enkelgeneration halbieren gegenwaertig aktivierende familienpolitik unterlassen demografische katastrophe bevoelkerungstransformation zwingen verfestigen quasiautomatische einbuergerung zeitablauf sawsan chebli zitat demografie faktum erdogan zitat macht kind anschlag staatsvolk feindlich uebernahme dame vorwandcharakter operation tarnname fluechtlingsschutz klarname auslaenderimport absurd gehaeuft inkaufnahme nachteil aufnahmeland zweistellige milliardenbetraege veranstaltung kollaps wohnungsmarkts zusammenbruch bildungssystems totalverlust inner aufnahmeanspruch mehrheitlich gering durchzug zig sicher drittstaaten asyl subsidiaerer status assad buergerkrieg million syrer einbuergern dame merkel spiel erfinden ampel steigern gefordert einbuergerungsautomatismus illegal millionenheeres laufen friedrich merz linke vorbei friedrich merz ehrenwert merz kuendigen unumstoesslich amtszeit ausnahmslos illegal einreise zurueckweisen person schutzanspruch monat gibt asylbewerbern zweimal zurueckweisungen friedrich merz ehrenwert merkel moralisch rueckgrat brechen machtgeiler opportunismus friedrich merz schuldenmacherei vorbei friedrich merz ehrenwert verscherbeln anbiedern staatsbuergerschaft einbuergerung ueberproportionale auslaenderkriminalitaet statistisch unsichtbar einbuergerung abschiebung dauerhaft zugang sozialsystemen hinwendung einbuergerung per fristablauf dame lebensgefuehl mensch freibad kita schule endlos gemobbt nachts bahnhof zerstoeren innenstaedte staatsbuergerschaftsrecht ausschliesslich linke unumkehrbar identitaet zerstoeren hauptsache cduler kanzler kind enkel verfluchen merkel ampel friedrich merz ehrenwert'

In [22]:
df_afd_2024_migration["text"].tolist()[487]

'Sehr geehrte Frau Präsidentin! Geschätzte Kolleginnen und Kollegen! Mit dem FAG-Änderungsgesetz werden bei der Finanzaufteilung zwischen Bund und Ländern viele kleine Schräubchen verstellt. Die Probleme unseres Landes löst das aber in keiner Weise.\n\nErstes Beispiel dafür ist die – so heißt es im Gesetz – „Bewältigung der Fluchtmigration“. Dafür sollen die Länder dieses Jahr 500 Millionen Euro mehr bekommen, zusammen dann 1,75 Milliarden Euro. Bewältigt wird damit aber gar nichts. Auch dieses Jahr ist mit 300\u202f000 neuen Asylanträgen zu rechnen. Die Migrationsgesamtkosten für Bund, Länder und Gemeinden liegen bei jährlich über 50 Milliarden Euro. Dabei kann man den Verlust der inneren Sicherheit und die wachsenden sozialen Konflikte überhaupt nicht mit Geld beziffern. Für alle Zweifler ein Beispiel: In der bayerischen Gemeinde Warngau mit knapp 4\u202f000 Einwohnern soll ein Containerlager für 500 Migranten entstehen. Das sorgt im Leben der Bürger für Stress, für Ängste und für Ko

'Werter Herr Präsident! Sehr geehrte Kolleginnen und Kollegen! Liebe Zuschauer! Als wir in der AfD-Fraktion erfahren haben, dass NOOTS im Plenum besprochen wird, war die ganze Fraktion elektrisiert. Wir haben aus fachfremden Arbeitskreisen Anfragen bekommen, ob wir Redeanteile abgeben können. Zwischenzeitlich mussten wir überlegen, ob wir siebenmal eine Minute oder doch lieber 14-mal 30 Sekunden dazu reden.\n\nGenau so hat es sich zugetragen oder so ähnlich, vielleicht war es auch ganz anders. Im Ergebnis habe ich jetzt sieben Minuten Redezeit zum Thema NOOTS. Und ich finde das toll!\n\nIch hoffe, Sie halten mich jetzt nicht für einen Nerd, wenn ich sage, dass das Thema NOOTS aus meiner Sicht ausgesprochen sexy ist.\n\nBevor ich das an einem Beispiel erkläre, möchte ich eine kurze Übersetzung geben: Das heißt so viel wie „Nationales Nur-einmal-erfassen-System“.\n\nIch will das am Beispiel des Kindergeldes erläutern; das ist hier ja schon angesprochen worden. Wenn man in Deutschland Kindergeld haben möchte, muss man als Allererstes eine Geburtsurkunde haben; ohne die gibt es kein Kindergeld. Eine Geburtsurkunde kriegt man idealerweise in der Außenstelle des Standesamtes in der Geburtsklinik, vorausgesetzt, das Kind kommt in einer Geburtsklinik zur Welt, und vorausgesetzt, diese Geburtsklinik hat eine Außenstelle des Standesamtes, und vorausgesetzt, diese Außenstelle hat zu der Zeit auch offen. Anderenfalls muss man zum Standesamt gehen. Das ist alles bewältigbar, ist unter Umständen aber relativ viel Arbeit. Wenn man die Geburtsurkunde hat, muss man als Nächstes zur Kindergeldstelle gehen. Die muss man ausfindig machen; das können unterschiedliche Stellen sein. Und da muss man noch mal einen ganz langen Antrag einreichen.\n\nWie wäre es denn, wenn man stattdessen im Krankenhaus einfach den Namen des Kindes und die Kontonummer fürs Kindergeld angeben würde und wenn man nach Hause kommt idealerweise schon die erste Kindergeldzahlung auf dem Konto wäre? Das klingt wie eine Utopie, ist es aber glücklicherweise nicht. Und um genau so was umzusetzen, brauchen wir NOOTS.\n\nDie Idee ist also: Wir erfassen ein Mal die Daten, und die werden intern weitergegeben; das ist ja gerade schon sehr gut erläutert worden. Die Idee ist nicht ganz neu. Als ich vor circa 30 Jahren meine ersten IT-Projekte geleitet habe, habe ich integrierte ERP-Systeme in Unternehmen des Mittelstandes eingeführt. Der große Vorteil dieser Systeme war, dass man den Auftrag nicht ausgedruckt und die Produktion ihn dann in ein anderes System eingetippt hat, sondern dass die Daten intern weitergereicht wurden. Vor 30 Jahren war das der Hit in der Privatwirtschaft. 30 Jahre später kommt es jetzt auch in der öffentlichen Verwaltung an. Das ist zwar spät, aber besser spät als nie an dieser Stelle.\n\nDas wird trotzdem kein Spaziergang, auch wenn dieses System in der Wirtschaft schon etabliert ist. Das sieht man allein schon daran, dass der NOOTS-Staatsvertrag nicht die einzige Grundlage ist, die wir brauchen, sondern es gibt noch so wohlklingende Gesetze wie das Identifikationsnummerngesetz, das Registermodernisierungsgesetz und das Onlinezugangsgesetz, die als Grundlage dafür dienen.\n\nDas zeigt schon, dass in der öffentlichen Verwaltung alles ein bisschen komplexer ist. Das kann man sicherlich noch deutlich vereinfachen. Aber man muss natürlich auch sagen: Da ist Sorgfalt absolut notwendig. – Wir reden hier über nicht weniger als über die digitale Identität der Bürger, um die es geht.\n\nSchon heute hat ein Identitätsdiebstahl dramatische Folgen, und man muss sich klarmachen: Er wird natürlich noch viel dramatischer, wenn wir ein System haben, wo alles untereinander ausgetauscht wird. Dann ist nämlich die komplette Identität weg, wenn sie geklaut wird. Dementsprechend sind zwei Dinge absolut notwendig: Es muss absolute Datensicherheit sichergestellt sein, und es muss auch ein Missbrauch des Staates verhindert werden.\n\nDer Weg dahin wird nicht einfach sein. Ich hatte schon mal die Ehre, an einem IT-Projekt teilzunehmen. Von 2016 bis 2021 war ich für den Berliner Bezirk Reinickendorf im Steuerungskreis des damaligen Digitalisierungsprojektes des rot-rot-grünen Senates vertreten. Der hatte sich vorgenommen, in sechs Jahren alle Verwaltungsvorgänge in Berlin zu digitalisieren – sehr ambitioniert. Von den 30 Mitgliedern hatte außer mir nur noch eine weitere Person Erfahrung in dem Bereich. Für alle anderen war es das erste Digitalisierungsprojekt. Entsprechend ist es auch gelaufen.\n\nIch habe dann – nur mal ein Beispiel – nach kurzer Zeit angemerkt, dass es mit dem Mitbestimmungs- und Datenschutzrecht in Berlin schwer werden wird, dieses Projekt überhaupt jemals umzusetzen. Das wurde dann weggewischt; ich hatte ja das falsche Parteibuch. Ein Jahr später hat dann die Projektleitung dem Steuerungskreis den Auftrag erteilt, zu prüfen, welche Änderungen im Mitbestimmungs- und Datenschutzrecht vorgenommen werden müssen, um den Projektvorgang zu beschleunigen.\n\nMan hatte es also ein Jahr lang liegen lassen. Umgesetzt wurde diese Maßnahme tatsächlich nie, wahrscheinlich weil irgendwann alle frustriert waren, da sie gemerkt haben, dass es nicht vorangeht.\n\nNun kann man aber aus gescheiterten Projekten sehr gut lernen, vor allem dann, wenn man nicht selber die Projekte in den Sand gesetzt hat, sondern andere; dann fällt es einem immer besonders leicht. Sieben Minuten reichen jetzt nicht aus, um über alle Fehler zu reden; aber das war einer.\n\nIch glaube, eine wichtige Sache kann man daraus mitnehmen: So ein Projekt läuft nicht in einer Legislaturperiode. Man sollte sich zwar ehrgeizige Ziele setzen, aber man muss sich klarmachen: Wir müssen schon mit fünf Legislaturperioden rechnen. Das heißt, es wird mehrere Regierungswechsel geben in dieser Zeit. Es macht also durchaus Sinn, das Projekt von Anfang an fraktionsübergreifend aufzusetzen, damit nicht mit jedem Regierungswechsel neu angefangen wird, sondern an diesem Projekt nahtlos weitergearbeitet werden kann.\n\nEs sei mir noch ein zweiter Hinweis erlaubt, den ich eingangs schon mal erwähnt habe: Ganz wesentlich ist die Akzeptanz in der Bevölkerung. Diese Akzeptanz werden wir nur kriegen, wenn wir einmal absolute Datensicherheit garantieren können. Es darf keinen Identitätsdiebstahl geben.\n\nWichtig ist außerdem: Dieses umfassende Mittel einer digitalen Identität darf nicht vom Staat missbraucht werden, indem sie zum Beispiel einfach gelöscht werden kann. Ich kann mir vorstellen, dass man dazu sogar das Grundgesetz ändern müsste, um hier einen Pflock einzuschlagen.\n\nDas Grundgesetz ist ja von den ursprünglichen Intentionen her ein Gesetz, das die Bürger vor einem übergriffigen Staat schützt. Damals, als noch der Horror des Nationalsozialismus ganz präsent war, hat man gesagt: Wir müssen die Bürger davor schützen. Dafür brauchen wir das Grundgesetz. – Heute wird es ja ganz anders benutzt.\n\nAn dieser Stelle: Ich drücke auf jeden Fall die Daumen, dass dieses Digitalisierungsprojekt erfolgreich ist. Ich denke, dass wir dafür einen fraktions- und legislaturperiodenübergreifenden Konsens benötigen. Die AfD ist gerne bereit, zu diesem beizutragen.'




















### Examining migration speech data from 1990-1993

In [23]:
df_reg_1990_migration = data[
    (data["Party"].isin(["CDU/CSU", "SPDCDU/CSU"])) & # Opposition filtern
    (data["date"].dt.year >= 1990) & # >= statt > um das Jahr 1990 mitzunehmen
    (data["date"].dt.year <= 1994)   # <= statt < um das Jahr 1994 mitzunehmen
].copy()

In [24]:
len(df_reg_1990_migration)

4522

In [86]:
df_reg_1990_migration["text_preprocessed_lemmatized"].tolist()[91]

'dame zeitlang narr narr narr aussage abraham lincoln unwillkuerlich unterlage nochmals jelpke gaukeln rassismus diskriminierung auslaendisch buergerin echte dame federfuehrend innenausschuss mitberatenden gruppe anwesend gaukeln tatsaechliche rechtliche gleichstellung auslaendisch buergerin barger rechtsordnung unterschiedlich staatsbuerger auslaender verzichten sachlich oeffentlich arbeitsmarkt differenzierung gaukeln festzustellenden auslaenderdiskriminierung begegnen ursache auslaenderfeindlichkeit unkenntnis angst diskriminierung tief kopf massnahme entgegenwirken bundestagsfraktion leimen effekthascherischen koalitionspartner gesetzentwurfes ernsthaft unterschiedlich deutschen lebend auslaender beibehalten integration positive veraenderung fortschritt moegen unterschiedlich groesse tempo bestreiten auslaenderrecht staatsangehoerigkeitsrecht demnaechst verabschiedend beanstanden offensive gewalt fremdenfeindlichkeit interessieren gruppierung gewaltpraevention jugendliche aufklaeru

'dame zeitlang narr narr narr aussage abraham lincoln unwillkuerlich unterlage nochmals jelpke gaukeln rassismus diskriminierung auslaendisch buergerin echte dame federfuehrend innenausschuss mitberatenden gruppe anwesend gaukeln tatsaechliche rechtliche gleichstellung auslaendisch buergerin barger rechtsordnung unterschiedlich staatsbuerger auslaender verzichten sachlich oeffentlich arbeitsmarkt differenzierung gaukeln festzustellenden auslaenderdiskriminierung begegnen ursache auslaenderfeindlichkeit unkenntnis angst diskriminierung tief kopf massnahme entgegenwirken bundestagsfraktion leimen effekthascherischen koalitionspartner gesetzentwurfes ernsthaft unterschiedlich deutschen lebend auslaender beibehalten integration positive veraenderung fortschritt moegen unterschiedlich groesse tempo bestreiten auslaenderrecht staatsangehoerigkeitsrecht demnaechst verabschiedend beanstanden offensive gewalt fremdenfeindlichkeit interessieren gruppierung gewaltpraevention jugendliche aufklaerungs integrationsmassnahmen entschieden vorgehen polizei justiz ursache bekaempfen buendelung massnahme stand offensive empfehlen beachtung abschliessend lade dame offensive gewalt fremdenfeindlichkeit mitwirken bedanken'

In [92]:
df_reg_1990_migration["text"].tolist()[93]

' Herr Kollege Singer, ich denke, daß das jetzt verabschiedete Verbrechensbekämpfungsgesetz, dem Sie zugestimmt haben, ergänzungsbedürftig ist. Wenn Sie mit der Polizei reden, dann ist ein Punkt natürlich von ausgesprochener Wesentlichkeit. Ich will das ganz vorurteilslos sagen, auch zu Herrn Kollege Dr. Hirsch: Wir müssen einmal darüber nachdenken, das zu tun, was Sie sicherlich gesagt haben und was wir auf dem Parteitag der CDU auch beschlossen haben, nämlich an die Umkehr der Beweislast heranzugehen.\nWenn wir das gemeinsam tun, dann wird dieses\nGesetz sicherlich noch effektiver sein. Ich wiederhole:\nWir haben Gesetze beschlossen, mit denen wir den\nKampf gegen die Gangster angesagt haben. Das wollen wir auch weiterhin tun.\nMeine Damen und Herren, ich darf die Liste der erfolgreichen Vorhaben fortsetzen: Wir haben das Ausländerrecht neu geordnet. Wir haben das Asylrecht novelliert. Zum Asylrecht: Unser Asylrecht ist weiterhin das großzügigste in der Welt, nur dem Mißbrauch wird e

' Herr Kollege Singer, ich denke, daß das jetzt verabschiedete Verbrechensbekämpfungsgesetz, dem Sie zugestimmt haben, ergänzungsbedürftig ist. Wenn Sie mit der Polizei reden, dann ist ein Punkt natürlich von ausgesprochener Wesentlichkeit. Ich will das ganz vorurteilslos sagen, auch zu Herrn Kollege Dr. Hirsch: Wir müssen einmal darüber nachdenken, das zu tun, was Sie sicherlich gesagt haben und was wir auf dem Parteitag der CDU auch beschlossen haben, nämlich an die Umkehr der Beweislast heranzugehen.\nWenn wir das gemeinsam tun, dann wird dieses\nGesetz sicherlich noch effektiver sein. Ich wiederhole:\nWir haben Gesetze beschlossen, mit denen wir den\nKampf gegen die Gangster angesagt haben. Das wollen wir auch weiterhin tun.\nMeine Damen und Herren, ich darf die Liste der erfolgreichen Vorhaben fortsetzen: Wir haben das Ausländerrecht neu geordnet. Wir haben das Asylrecht novelliert. Zum Asylrecht: Unser Asylrecht ist weiterhin das großzügigste in der Welt, nur dem Mißbrauch wird ein Riegel vorgeschoben, und wirklich politisch Verfolgten ist Schutz sicher. Aber, Herr Kollege Körper, ich weiß doch, wie schwer es gerade Ihrer Fraktion gefallen ist, diesem Gesetz letzten Endes mit einer Handvoll Stimmen zuzustimmen. Ich bedanke mich dafür. Aber Sie können doch auf Grund der Mehrheit im Bundesrat nicht sagen: Wer regiert hier denn eigentlich?\nIch meine, unser Asylrecht ist ein großzügiges Recht. Das wird auch durch die Entscheidung des Bundesinnenministers hinsichtlich der Abschiebung der Kurden bestätigt. Damit eines klar ist: Die türkischen Urteile gegen ihre eigenen kurdischen Abgeordneten sind ein schlimmes Zeichen für die Einschätzung der demokratischen Ordnung in der Türkei; sie verlangen eine neue Bewertung des Verhältnisses zu diesem NATO-Partner.\nAber ich denke, daß der Abschiebestopp bis zum 20. Januar genügend Gelegenheit gibt, darüber nachzudenken. Deswegen entbehren alle Vorwürfe an den Innenminister jeder Grundlage.\nDenn bisher — Sie wissen dies — ist kein einziger PKK-Anhänger von Deutschland an die Türkei ausgeliefert worden. Das ist auch die Linie des Innenministers. Wir wollen im Einzelfall entscheiden. Eine Einzelfallentscheidung ist letzten Endes generellen Regelungen vorzuziehen.\nMeine Damen und Herren, im Bereich der inneren Sicherheit müssen wir das Ausländer- und Asylrecht weiterentwickeln; denn der Friede, das wissen wir, beginnt im eigenen Haus. Im innenpolitisch geistigen Kampf um die Herrschaft muß die Gesinnung der Friedlosigkeit, die die Gewalt wollen würde, wenn sie nur könnte, verschwinden. Ich meine, der Etat des Finanzministers zeigt auf, daß wir im Bereich der Gewährleistung der inneren Sicherheit zufrieden sein können.\nIch denke erstens daran: Wir werden bis 1996 die Lücke im Bundesgrenzschutz geschlossen haben. Herzlichen Dank, Herr Bundesfinanzminister, herzlichen Dank, Herr Bundesinnenminister! Wir werden damit den Schleppern den Kampf angesagt haben. Das Schleppertum ist ein schlimmes Verbrechen. Wir müssen deswegen die Zahl an Grenzschutzbeamten erhöhen.\nAls zweites werden wir das Bundeskriminalamt personell und finanziell besser ausstatten. Ich meine, die Verbrechensbekämpfung beim, BKA hat sich bewährt.\nSchließlich wollen wir weiterhin die Länder bei den Bereitschaftspolizeien unterstützen. Das kostet Geld. Das wissen wir. Wir wissen auch, daß für die innere Sicherheit in erster Linie die Länder verantwortlich\nsind, in denen Sie oft das Sagen haben. Ich weiß, daß wir als Bund Gesetze zu beschließen haben, und wir werden dies tun.\nMeine Damen und Herren, wir brauchen z. B. dringend eine gesetzliche Regelung der Hauptverhandlungshaft. Warum haben Sie dem im Vermittlungsausschuß nicht zugestimmt? Sie wollen darüber nachdenken. Unser Ziel: Sofort festzunehmen, sofort zu verurteilen, sofort zum Strafantritt zu kommen. So Dinge wie in Oberhof dürfen sich nicht wiederholen. Das hat diese Koalition beschlossen.\nIch fordere Sie auf, darüber nachzudenken und es uns gleichzutun. Wir brauchen — richtig, Herr Kollege Körper — eine Novellierung des BKA-Gesetzes. Da ist eine effektive Verbrechensbekämpfung erforderlich. Ich verstehe nicht, warum die großen Länder sich dagegen sperren, dem BKA die Vorfeldbeobachtung zu übertragen. Aber dies, meine Damen und Herren, wird nicht ausreichen.\nWas ich Ihnen vorschlage, ist etwas, was wir bereits als Koalition beschlossen hatten. Ich halte es für dringend vonnöten, das G-10-Gesetz zu erweitern. Wir setzen den Bundesnachrichtendienst zum Einsatz gegen Terrorismus, gegen Drogen und gegen Handel mit spaltbarem Material ein. Warum beziehen wir nicht die individuelle Verbrechensbekämpfung ein? Es kann doch nicht sein, daß ein Gangster hier überwacht wird, weil er mit Drogen oder mit radioaktivem Material handelt; dieser Gangster hat eine zweite Wohnung in Frankreich, und er zieht dorthin und betreibt sein verbrecherisches Geschäft von dort. Dieser Gangster muß überwacht werden, meine Damen und Herren. Ich bitte Sie herzlich, das G10-Gesetz mit uns entsprechend zu ändern.\n— Ich spreche ja nicht von Ihnen, Herr Kollege Fischer. Ich spreche von Gangstern. Wissen Sie, Sie werden noch Gelegenheit haben, mit uns gemeinsam unverkrampft, ohne Überspanntheiten und ohne Übertreibungen an die Reformierung von Gesetzen zu gehen.\n'


## Constructing an pro-migration lexicon

### 2010-2015

In [26]:
relevante_parteien = ["GRÜNE", "LINKE.", "SPD", "CDU/CSU"]

df_migration_filtered_2010_2015 = data[
    (data["Party"].isin(relevante_parteien)) & 
    (data["date"].dt.year >= 2010) & 
    (data["date"].dt.year <= 2015)
].copy()

len(df_migration_filtered_2010_2015)

21909

In [132]:
df_migration_filtered_2010_2015["text_preprocessed_lemmatized"].tolist()[756]

'dame krise be merkbar bundesamt migration fluechtling massiv anstieg asylbewer berzahlen asylantraege strategie stabilisierung asylsystems zentrale schutzberechtigten integrieren unberechtigte asylantraege schnell abschliessen zahlreiche massnahme bun desamt migration fluechtling stel len aufstocken westbalkanlaender sicher herkunftsstaaten unbegruendet schnell abschliessen koen nen kommune million versorgung bringung fluechtling entlasten novellierte asylbewerberleistungsgesetz entlastung struktur integration fluechtling arbeitsmarktzugang erleichtern residenzpflicht rangpruefung einschraenken zuegig integrieren fin asylsystem nachhaltig stabilisieren mues sen gross unberechtigt asylan traege spuerbar reduzieren asylantraege balkanstaaten regis triert ablehnungsquote asylantraege sy rischen fluechtling selben zeitraum halb unberechtigt asylan traege zuegig zurueckfuehren nachah mer abhalten geld kriminelle schleuser verschwenden balkanstaaten asyl migrationssystem klaeren legal arbei



'dame krise be merkbar bundesamt migration fluechtling massiv anstieg asylbewer berzahlen asylantraege strategie stabilisierung asylsystems zentrale schutzberechtigten integrieren unberechtigte asylantraege schnell abschliessen zahlreiche massnahme bun desamt migration fluechtling stel len aufstocken westbalkanlaender sicher herkunftsstaaten unbegruendet schnell abschliessen koen nen kommune million versorgung bringung fluechtling entlasten novellierte asylbewerberleistungsgesetz entlastung struktur integration fluechtling arbeitsmarktzugang erleichtern residenzpflicht rangpruefung einschraenken zuegig integrieren fin asylsystem nachhaltig stabilisieren mues sen gross unberechtigt asylan traege spuerbar reduzieren asylantraege balkanstaaten regis triert ablehnungsquote asylantraege sy rischen fluechtling selben zeitraum halb unberechtigt asylan traege zuegig zurueckfuehren nachah mer abhalten geld kriminelle schleuser verschwenden balkanstaaten asyl migrationssystem klaeren legal arbeitskraft aussichts asylantrag bruchteil ausreisepflichti gen auslaender abschieben wa ren geduldet registrieren abgeschoben per sonen abschiebung zustaendig verfahrensregeln verantworten umfangreiche asylsystem auslaender gedulden erfolgreich integrieren al ters stichtagsunabhaengiges bleiberecht bekom men integrieren acht sprachkenntnisse verfuegen le bensunterhalt ueberwiegen inte grierte jugendliche aehnlich voraus setzungen dauerhaft bleiberecht aufenthaltsgesetz aufenthaltstitel ermoeglichen auslaendische berufsqualifikation fort bildungsmassnahmen vollstaendig anerkennen bundesinnenminister zielen auslaenderrecht erleichtern zuwanderung fachkraeften dreistufige mussregelung ausweisungsrecht grundlegend reformieren gericht klage reine ermessensent scheidungen reagieren wandel rechtsprechung gericht beschleuni gen behoerde entweder bestaetigen ersetzen behoerde schnell rechtssi cherheit asylbewerber ausweisen ausweisungs bleibeinteressen gewichten bleibeinteresse wiegen minderjaehrige auslaender gebaeren kind ausweisungsinteresse wiegen auslaender hass gewalt rufen terror freiheits strafen verurteilen abschiebehaft kriterium de finieren fluchtgefahr kolle gin jelpke dublin iii bgh bgh verankerung national fehlen entspre chende fluchtgefahr indiz mand zugriff behoerde entziehen identitaet taeuschen mitwirkung verweigern indiz fluchtgefahr moegen mitzu wirken ausweisung rechts reagieren asylbewerber einzelfallpruefung willkuer ausschliessen haft maessigen rechtsstaat reichend lieber ruediger veit abend umgang unbegleitet minderjaehri gen fluechtling debattieren geduldeter ausbildung handwerks betrieb abge schieben absatz aufenthaltsgesetzes besagen per soenlichen dauer ausbildung duldung verwahren handwerksbetriebe ausbil dungsbetriebe jung asylbewerber ver ankern'














In [131]:
df_migration_filtered_2010_2015["text"].tolist()[756]

'\nSehr geehrte Frau Präsidentin! Meine sehr geehrten\n\nDamen und Herren! Die weltweiten Krisen machen sich\nauch in diesem Jahr in Deutschland nach wie vor bemerkbar. Das Bundesamt für Migration und Flüchtlinge\nerwartet erneut einen massiven Anstieg der Asylbewerberzahlen von zuletzt 203 000 auf 300 000 Asylanträge\nin diesem Jahr.\n\nDie Strategie der Großen Koalition zur Stabilisierung\nunseres Asylsystems hat zwei zentrale Ziele: Erstens.\nDie Schutzberechtigten sollen besser integriert werden.\nZweitens. Unberechtigte Asylanträge sollen schneller\nabgeschlossen werden. Bereits im letzten Jahr haben wir\nzahlreiche Maßnahmen umgesetzt. Wir haben im Bundesamt für Migration und Flüchtlinge die Zahl der Stellen um rund 30 Prozent aufgestockt. Wir haben drei\nWestbalkanländer zu sicheren Herkunftsstaaten erklärt,\num unbegründete Anträge schneller abschließen zu können. Wir haben in diesem Jahr Länder und Kommunen\num 556 Millionen Euro bei der Versorgung und Unterbringung von Flücht





'\nSehr geehrte Frau Präsidentin! Meine sehr geehrten\n\nDamen und Herren! Die weltweiten Krisen machen sich\nauch in diesem Jahr in Deutschland nach wie vor bemerkbar. Das Bundesamt für Migration und Flüchtlinge\nerwartet erneut einen massiven Anstieg der Asylbewerberzahlen von zuletzt 203 000 auf 300 000 Asylanträge\nin diesem Jahr.\n\nDie Strategie der Großen Koalition zur Stabilisierung\nunseres Asylsystems hat zwei zentrale Ziele: Erstens.\nDie Schutzberechtigten sollen besser integriert werden.\nZweitens. Unberechtigte Asylanträge sollen schneller\nabgeschlossen werden. Bereits im letzten Jahr haben wir\nzahlreiche Maßnahmen umgesetzt. Wir haben im Bundesamt für Migration und Flüchtlinge die Zahl der Stellen um rund 30 Prozent aufgestockt. Wir haben drei\nWestbalkanländer zu sicheren Herkunftsstaaten erklärt,\num unbegründete Anträge schneller abschließen zu können. Wir haben in diesem Jahr Länder und Kommunen\num 556 Millionen Euro bei der Versorgung und Unterbringung von Flüchtlingen entlastet. Im Jahr 2016 wird\ndas novellierte Asylbewerberleistungsgesetz zu noch\nmehr Entlastungen führen.\n\n\nGleichzeitig haben wir die Strukturen zur Integration\nvon Flüchtlingen verbessert. Der Arbeitsmarktzugang\nwurde erleichtert, und die Residenzpflicht und die Vorrangprüfung wurden eingeschränkt. Wer bei uns Schutz\nbekommt, der soll sich zügig integrieren und Arbeit finden können.\n\nUm das Asylsystem nachhaltig zu stabilisieren, müssen wir aber die große Zahl der unberechtigten Asylanträge spürbar reduzieren. Allein im Januar dieses Jahres\nwurden 11 700 Asylanträge aus den Balkanstaaten registriert, obwohl die Ablehnungsquote in diesen Fällen bei\nfast 100 Prozent liegt. Die Zahl der Asylanträge von syrischen Flüchtlingen war im selben Zeitraum nicht einmal halb so hoch.\n\nDie Zahl der offensichtlich unberechtigten Asylanträge muss zügiger zurückgeführt werden, um Nachahmer davon abzuhalten, Geld an kriminelle Schleuser zu\nverschwenden. Gerade in den Balkanstaaten müssen wir\nnoch besser über unser Asyl- und Migrationssystem aufklären; denn manch einer könnte auf ganz legalem Weg\nals Arbeitskraft zu uns kommen, statt einen aussichtslosen Asylantrag zu stellen.\n\nSeit Jahren wird nur ein Bruchteil der ausreisepflichtigen Ausländer tatsächlich abgeschoben. Ende 2014 waren 113 221 Geduldete hier in Deutschland registriert.\nAbgeschoben wurden im letzten Jahr lediglich 10 800 Personen.\n\nJa, für die Abschiebung sind die Länder zuständig,\naber der Bund hat die Verfahrensregeln zu verantworten.\nAn diesem Punkt setzt der vorliegende Gesetzentwurf an\nund sieht umfangreiche Verbesserungen im Asylsystem\nvor. Ausländer, die schon lange in Deutschland geduldet\nsind und sich erfolgreich integriert haben, sollen ein alters- und stichtagsunabhängiges Bleiberecht bekommen. Als gut integriert gilt jemand, der seit acht Jahren\nhier lebt, über Sprachkenntnisse verfügt und seinen Lebensunterhalt überwiegend selbst sichern kann. Gut integrierte Jugendliche unter 21 sollen bei ähnlichen Voraussetzungen bereits nach vier Jahren ein dauerhaftes\nBleiberecht erhalten können.\n\nMit § 17 a Aufenthaltsgesetz schaffen wir einen\nneuen Aufenthaltstitel in Deutschland, der es ermöglicht,\ndie ausländische Berufsqualifikation bei uns durch Fortbildungsmaßnahmen vollständig anerkennen zu lassen.\nDamit verbessert der Bundesinnenminister ganz gezielt\ndas Ausländerrecht und erleichtert die Zuwanderung von\nFachkräften.\n\n\nGleichzeitig soll die bisherige dreistufige Kann-, Sollund Mussregelung im Ausweisungsrecht grundlegend\nreformiert werden. Das ist richtig. Die Gerichte haben\nbei den meisten Klagen ohnehin reine Ermessensentscheidungen getroffen. Wir reagieren damit auf den\nWandel in der Rechtsprechung. Die Gerichte werden in\nZukunft – das wird die Verfahren erheblich beschleunigen – die Entscheidung der Behörde entweder bestätigen\noder ersetzen. Es wird also nicht an die Behörde zurückverwiesen. Auch damit werden wir schneller Rechtssicherheit für die Asylbewerber schaffen, ob sie bleiben\nkönnen, weil sie einen entsprechenden Anspruch haben,\noder ob sie ausgewiesen werden müssen.\n\nWir werden klare Ausweisungs- und Bleibeinteressen\nformulieren und gewichten. Auch das ist richtig. Ein\nBleibeinteresse wiegt zum Beispiel besonders schwer\nbei Minderjährigen und bei Ausländern, die in Deutschland geboren wurden oder hier eigene Kinder haben. Das\nAusweisungsinteresse wiegt zum Beispiel besonders\nschwer, wenn ein Ausländer zu Hass oder Gewalt auf\n\n\nruft, den Terror unterstützt oder zu längeren Freiheitsstrafen verurteilt wurde. Auch das ist richtig.\n\nWir werden für die Abschiebehaft klare Kriterien definieren, wann von einer Fluchtgefahr ausgegangen werden kann. Damit machen wir nichts anderes, Frau Kollegin Jelpke, als die Dublin-III-Verordnung und den\nBeschluss des BGH vom Juni 2014 umzusetzen. Der\nBGH hat nämlich festgestellt, dass die Verankerung im\nnationalen Recht fehlt. Wir sind daher gehalten, entsprechende Regelungen zu formulieren.\n\nWann geht man von einer Fluchtgefahr aus? Es gibt\nzunächst einmal Indizien. Wenn sich zum Beispiel jemand dem Zugriff der Behörden entziehen will, über\nseine Identität täuscht oder die Mitwirkung verweigert,\ndann sind das erst einmal Indizien für eine Fluchtgefahr,\ndie aber wohl begründet ist. Denn derjenige, der bei uns\nbleiben möchte, hat an entsprechenden Verfahren mitzuwirken; er hat sich zu beteiligen. Auch das ist bei der\nAusweisung von Interesse. Auch hier muss der Rechtsstaat reagieren können, wenn das von dem Asylbewerber\nnicht erfüllt wird.\n\nIm Übrigen bleibt es bei der Einzelfallprüfung. Damit\nwird Willkür ausgeschlossen; die Haft muss verhältnismäßig sein. Ich denke, hier sorgt der Rechtsstaat für ausreichende Sicherheit.\n\n\nLieber Kollege Rüdiger Veit, wir haben gestern\nAbend über den Umgang mit unbegleiteten minderjährigen Flüchtlingen debattiert. In der Debatte wurde gesagt,\ndass jemand, der als Geduldeter eine Ausbildung in\nDeutschland macht, zum Beispiel in einem Handwerksbetrieb, die ganze Zeit damit rechnen muss, dass er abgeschoben wird. Nein, das stimmt nicht. § 60 a Absatz 2\nSatz 3 des Aufenthaltsgesetzes besagt, dass es aus persönlichen Gründen möglich ist, für die gesamte Dauer\nder Ausbildung bei uns eine Duldung zu erhalten. Es ist\nalso bereits möglich.\n\n\n– Ja. Ich verwahre mich aber dagegen, dass immer wieder gesagt wird, die Handwerksbetriebe bzw. Ausbildungsbetriebe könnten keine jungen Asylbewerber einstellen.\n\n\nNatürlich ist das nach unserem Gesetz bereits möglich.\nDas muss nur noch von den Ländern entsprechend verankert werden.\n\n'




### 2015-2018

In [28]:
relevante_parteien = ["GRÜNE", "LINKE.", "SPD", "CDU/CSU"]

df_migration_filtered_2015_2018 = data[
    (data["Party"].isin(relevante_parteien)) & 
    (data["date"].dt.year >= 2015) & 
    (data["date"].dt.year <= 2018)
].copy()

len(df_migration_filtered_2015_2018)

12726

In [40]:
df_migration_filtered_2015_2018["text_preprocessed_lemmatized"].tolist()[485]

'lieben rechte menschenrechtsverletzungen bewirken be troffenen gross leid auge men schenrechte meinungs presse religionsfrei heit koerperlich unversehrtheit darue ber hinwegtaeuschen jahrhundert global achtung menschenrechten oftmals traum realitaet auge erhalt men schenrechten tuerkei autoritaer vormarsch begeben fluechtlingsstroeme million mensch flucht dilem ma aleppo mensch fluechten krisenauswirkungen auswirkung populistische ge ben nahrung auftrieb populisten hetzen fremde schueren angst befoerdern neiddebatten umso rechtsnormen menschenrech te schuetzen kritisch wirksamkeit ueberpruefen se zweier fundiert berich te eu deutsche institut menschenrechte deutsche institut menschenrechte rassismus vormarsch aufpassen men schenrechtsvertraege rassismus po litischen oeffentlich entge genzutreten staatlich verantwortungstraegerinnen traeger politikerin auffordern explizit rassistisch aeusserung tate betrachte ernstzuneh menden hinweis etablieren par teien sprache wahlprogrammen naehe p

'lieben rechte menschenrechtsverletzungen bewirken be troffenen gross leid auge men schenrechte meinungs presse religionsfrei heit koerperlich unversehrtheit darue ber hinwegtaeuschen jahrhundert global achtung menschenrechten oftmals traum realitaet auge erhalt men schenrechten tuerkei autoritaer vormarsch begeben fluechtlingsstroeme million mensch flucht dilem ma aleppo mensch fluechten krisenauswirkungen auswirkung populistische ge ben nahrung auftrieb populisten hetzen fremde schueren angst befoerdern neiddebatten umso rechtsnormen menschenrech te schuetzen kritisch wirksamkeit ueberpruefen se zweier fundiert berich te eu deutsche institut menschenrechte deutsche institut menschenrechte rassismus vormarsch aufpassen men schenrechtsvertraege rassismus po litischen oeffentlich entge genzutreten staatlich verantwortungstraegerinnen traeger politikerin auffordern explizit rassistisch aeusserung tate betrachte ernstzuneh menden hinweis etablieren par teien sprache wahlprogrammen naehe parole begeben stattdessen se parole wenden aufzeigen fluechtlingskrise massnahme ergreifen wirken beschleunigten asylverfahren finanzielle stuetzung kommune beschleunigung integrationsprozesse mensch sozialdemokratin sozialdemokrat stolz frank schwabe nap familienzusammenfuehrung absolut akzeptabel menschenunwuerdig familien angehoerig jahrelang warten akut heinrich massnah men handlungsbedarf ha ben familienzusammenfuehrung massnahme steuern ansetzen armut kind alleinerziehende zusehen vorgelegt zeitnah signal armut bedrohen kind zeichen populisten unwahrheit behaupten hiesig geld uebrig fluechtling aufbrauchen massnahme ansetzen eu menschenrechten eu einbeziehen akteur menschenrechte heinrich tom koenigs deutscher auffuehren erken nen herausforderung vielfaeltig fassen sodass alleine stemmen effektiv eu europaeisch wertege leiten aussenpolitik verschreiben menschenrechte ruf populisten protektionistisch ausrichten absage erteilen eu klarmachen waehrungsuni on friedens werteunion mitgliedstaaten eu ganzes wuensche hinwirken liegend weihnachtszeit verwenden ruhe ruhe einsatz staerkung menschenrechte schoepfen dank'


In [39]:
df_migration_filtered_2015_2018["text"].tolist()[485]

'\nSehr geehrter Herr Präsident! Liebe Kolleginnen und\n\nKollegen! Warum reden wir heute hier über Menschenrechte? Menschenrechtsverletzungen bewirken bei Betroffenen großes Leid, das müssen wir uns immer vor\nAugen halten . In Deutschland und in Europa sind Menschenrechte wie Meinungs-, Presse- oder Religionsfreiheit, aber auch das Recht auf körperliche Unversehrtheit\nweitgehend selbstverständlich . Das darf aber nicht darüber hinwegtäuschen, dass auch im 21 . Jahrhundert die\nglobale Achtung von Menschenrechten oftmals mehr ein\nTraum ist als Realität .\n\nGerade die letzten beiden Jahre haben uns vor Augen\ngeführt, wie wichtig es ist, sich für den Erhalt von Menschenrechten einzusetzen . Ich denke da an die Türkei, die\nsich mit immer autoritärerer Politik auf den Vormarsch\nbegibt, oder auch an die weltweiten Flüchtlingsströme .\nMillionen von Menschen sind auf der Flucht . Das Dilemma in Aleppo wurde angesprochen . Viele der Menschen,\ndie flüchten, sind zu uns nach Deutschland

'\nSehr geehrter Herr Präsident! Liebe Kolleginnen und\n\nKollegen! Warum reden wir heute hier über Menschenrechte? Menschenrechtsverletzungen bewirken bei Betroffenen großes Leid, das müssen wir uns immer vor\nAugen halten . In Deutschland und in Europa sind Menschenrechte wie Meinungs-, Presse- oder Religionsfreiheit, aber auch das Recht auf körperliche Unversehrtheit\nweitgehend selbstverständlich . Das darf aber nicht darüber hinwegtäuschen, dass auch im 21 . Jahrhundert die\nglobale Achtung von Menschenrechten oftmals mehr ein\nTraum ist als Realität .\n\nGerade die letzten beiden Jahre haben uns vor Augen\ngeführt, wie wichtig es ist, sich für den Erhalt von Menschenrechten einzusetzen . Ich denke da an die Türkei, die\nsich mit immer autoritärerer Politik auf den Vormarsch\nbegibt, oder auch an die weltweiten Flüchtlingsströme .\nMillionen von Menschen sind auf der Flucht . Das Dilemma in Aleppo wurde angesprochen . Viele der Menschen,\ndie flüchten, sind zu uns nach Deutschland gekommen.\n\nDiese Krisenauswirkungen haben auch Auswirkungen\nauf populistische Kräfte in unserem eigenen Land . Sie geben ihnen neue Nahrung und leider auch neuen Auftrieb .\nDie Populisten hetzen gegen Fremde . Sie schüren Ängste\nund befördern Neiddebatten . Umso wichtiger ist es, dass\nwir die vielen Rechtsnormen, die unsere Menschenrechte schützen, immer wieder kritisch auf ihre Wirksamkeit\nhin überprüfen . Deswegen bin ich sehr froh, dass wir diese Debatte heute auf Grundlage zweier fundierter Berichte, des Berichts der EU und des Berichts des Deutschen\nInstituts für Menschenrechte, führen können .\n\nKonkret weist der Bericht des Deutschen Instituts\nfür Menschenrechte darauf hin, dass der Rassismus in\nDeutschland auf dem Vormarsch ist und dass wir hier\naufpassen müssen . In dem Bericht steht ganz konkret:\n\nDeutschland ist durch die internationalen Menschenrechtsverträge verpflichtet, Rassismus im politischen Raum und im öffentlichen Leben entgegenzutreten …\n\nStaatliche Verantwortungsträgerinnen und -träger sowie Politiker und Politikerinnen werden aufgefordert,\n\nsich explizit gegen rassistische Äußerungen und Taten\nauszusprechen . Ich betrachte das als einen ernstzunehmenden Hinweis, dass sich gerade die etablierten Parteien in ihrer Sprache, aber auch in ihren Partei- und\nWahlprogrammen nicht in die Nähe rechter Parolen und\nInhalte begeben dürfen .\n\n\nStattdessen ist es ganz wichtig, dass wir uns gegen diese Parolen wenden und aufzeigen, dass wir in Zeiten der\nFlüchtlingskrise Maßnahmen ergriffen haben, die nach\nund nach wirken . Ich denke da beispielsweise an die beschleunigten Asylverfahren oder an die finanzielle Unterstützung des Bundes für die Länder und die Kommunen\nzur Beschleunigung der Integrationsprozesse . Wir haben\nauch nicht vergessen, was für unsere eigene Bevölkerung\nwichtig ist . So haben wir beispielsweise viele Gesetze\nauf den Weg gebracht, die das Leben vieler Menschen\nin unserem Land sozial gerechter machen . Darauf sind\ngerade wir Sozialdemokratinnen und Sozialdemokraten\nsehr stolz .\n\n\nEs gibt noch vieles mehr, was wir tun können und\nauch tun müssen . Kollege Frank Schwabe hat den NAP\nangesprochen, hat die Familienzusammenführung angesprochen . Auch ich sage ganz deutlich: Es ist absolut\nnicht akzeptabel und menschenunwürdig, dass Familienangehörige jahrelang darauf warten müssen, zusammengeführt zu werden .\n\n\nIn diesem Bereich müssen wir akut mehr tun .\n\nKollege Heinrich, Sie haben sehr viele gute Maßnahmen angesprochen, bei denen wir Handlungsbedarf haben . Die Familienzusammenführung ist eine Maßnahme,\ndie wir als Koalition ganz konkret steuern können . Deswegen müssen wir hier ansetzen .\n\n\nMit Blick auf die von Armut betroffenen Kinder von\nAlleinerziehenden in unserem Land sage ich: Auch hier\nhaben wir die Möglichkeit, ganz konkret etwas zu unternehmen . Wir müssen zusehen, dass wir das vorgelegte\nKonzept zeitnah umsetzen . Das wäre ein ganz wichtiges\nSignal für die von Armut bedrohten Kinder .\n\n\nNatürlich wäre es auch wichtig, damit ein deutliches\nZeichen zu setzen, dass die Populisten die Unwahrheit\nsagen, wenn sie behaupten, für die hiesige Bevölkerung\nbliebe kein Geld mehr übrig, weil alles für Flüchtlinge\naufgebraucht würde . Auch hier können wir mit konkreten\nMaßnahmen ansetzen .\n\nEin letztes Wort zur EU . Wenn wir über den Schutz\nvon Menschenrechten sprechen, müssen wir immer auch\ndie EU einbeziehen . Sie ist ein wichtiger Akteur für mehr\nMenschenrechte in der Welt . Sie tut viel; Herr Heinrich,\n\nTom Koenigs\n\nDeutscher Bundestag – 18 . Wahlperiode – 210 . Sitzung . Berlin, Freitag, den 16 . Dezember 2016 21069\n\n\n\nSie haben einiges aufgeführt . Wir müssen einfach erkennen, dass die Herausforderungen sehr vielfältig und umfassend sind, sodass ein Land alleine sie nicht stemmen\nkann .\n\nDeshalb ist es ganz wichtig und viel effektiver, dass\nwir uns als Teil der EU einer europäischen und wertegeleiteten Außenpolitik verschreiben und sie unterstützen;\ndenn das ist für den Schutz der Menschenrechte in der\nWelt wichtig . Dem Ruf der Populisten, nur das eigene\nLand in den Blick zu nehmen und sich protektionistisch\nauszurichten, muss eine klare Absage erteilt werden .\n\n\nDie EU ist nicht nur – auch das müssen wir immer\nwieder klarmachen – eine Wirtschaft- und Währungsunion, sondern sie ist auch – das sind zwei ganz wesentliche\nPunkte – eine Friedens- und Werteunion . Das muss für\ndie einzelnen Mitgliedstaaten deutlich werden, aber auch\nfür die EU als Ganzes .\n\nIch wünsche mir, dass wir alle darauf hinwirken und\ndie vor uns liegende Weihnachtszeit dafür verwenden,\ndass wir selbst zur Ruhe kommen, aber auch dazu, dass\nwir aus dieser Ruhe Kraft für einen weiteren Einsatz zur\nStärkung der Menschenrechte auf der Welt schöpfen .\n\nVielen Dank .\n\n\n'

### 2021-2022

In [29]:

relevante_parteien = ["GRÜNE", "LINKE.", "SPD", "CDU/CSU"]

df_migration_filtered_2021_2022 = data[
    (data["Party"].isin(relevante_parteien)) & 
    (data["date"].dt.year >= 2021) & 
    (data["date"].dt.year <= 2022)
].copy()
